# FreightQuote AI — Milestone 4: Streamlit Application Gateway
This notebook runs the complete FreightQuote AI Milestone 4 Streamlit application in Google Colab. It writes all consolidated modules and sets up a public tunnel (localtunnel / ngrok) for presentation.

### Step 1: Install Required Dependencies
Installs all computational, modeling, and UI packages used by the platform.

In [ ]:
!pip install streamlit rank_bm25 pymupdf reportlab streamlit_option_menu plotly joblib pandas numpy torch transformers pyjwt bcrypt langchain langchain-community sentence-transformers faiss-cpu

### Step 2: Establish Working Directories & Drive Connection
Creates working folders. You can mount Google Drive to persist the SQLite database (`freightquote.db`) and models.

In [ ]:
import os
from google.colab import drive

BASE_DIR = "/content/freightquote_m4"
os.makedirs(BASE_DIR, exist_ok=True)

st_drive = input("Would you like to mount Google Drive to persist database/models? (y/n): ").strip().lower()
if st_drive == 'y':
    drive.mount('/content/drive')
    db_drive_path = "/content/drive/MyDrive/freightquote.db"
    if os.path.exists(db_drive_path):
        import shutil
        shutil.copy(db_drive_path, os.path.join(BASE_DIR, "freightquote.db"))
        print("Copied existing freightquote.db from Google Drive.")

### Step 3: Write Configuration Utilities Module
Writes `config_utils.py` containing database connections, ports registry, and secrets loader.

In [ ]:
%%writefile /content/freightquote_m4/config_utils.py
"""
config_utils.py — Configuration, Ports Registry, and Database Utilities for FreightQuote AI.
Consolidates config.py, ports.py, database.py, and JWT auth utilities.
"""
import os
import json
import sqlite3
import datetime
import hashlib
import jwt
from typing import Dict, Any, List, Tuple

# =============================================================================
# 1. ENVIRONMENT CONFIGURATION & SECRETS
# =============================================================================
def get_secret(name: str) -> str:
    # Try loading from session secrets file first
    if os.path.exists(".session_secrets.json"):
        try:
            with open(".session_secrets.json", "r") as f:
                data = json.load(f)
                val = data.get(name)
                if val:
                    return val
        except Exception:
            pass

    # Fallback to Google Colab userdata
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return ""

STORAGE_DIR = "."
MODELS_DIR = os.path.join(STORAGE_DIR, "models")
RAG_DIR = os.path.join(STORAGE_DIR, "rag_documents")
FAISS_DIR = os.path.join(STORAGE_DIR, "faiss_index")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RAG_DIR, exist_ok=True)
os.makedirs(FAISS_DIR, exist_ok=True)

DATABASE = os.path.join(STORAGE_DIR, "freightquote.db")

AGENT1_MODEL_PATH = os.path.join(MODELS_DIR, "freight_price_model.pkl")
AGENT2_MODEL_PATH = os.path.join(MODELS_DIR, "route_delay_model.pkl")
AGENT3_MODEL_PATH = os.path.join(MODELS_DIR, "carrier_compliance_model.pkl")

# Credentials & API Keys
JWT_SECRET_KEY = get_secret("JWT_SECRET") or "freightquote-cyber-secure-jwt-key"
HF_TOKEN = get_secret("HF_TOKEN")
EMAIL_ADDRESS = get_secret("EMAIL_ADDRESS")
EMAIL_PASSWORD = get_secret("EMAIL_PASSWORD")
ADMIN_EMAIL_ID = get_secret("ADMIN_EMAIL_ID") or "admin@freightquote.ai"
ADMIN_PASSWORD = get_secret("ADMIN_PASSWORD") or "admin@123"
NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN") or get_secret("NGROK_AUTH_TOKEN")

# LLM & Embeddings Settings
QWEN_MODEL = "Qwen/Qwen2.5-3B-Instruct"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
VECTOR_DB_PATH = FAISS_DIR
KNOWLEDGE_FOLDER = RAG_DIR

# =============================================================================
# 2. PORT REGISTRY (SINGLE SOURCE OF TRUTH)
# =============================================================================
PORTS = {
    "Mumbai (JNPT)": {
        "id": "INBOM", "name": "Mumbai (JNPT)", "country": "India", "region": "South Asia",
        "latitude": 18.95, "longitude": 72.82, "congestion": "Medium", "dwell_time": 2.8, "vessel_count": 42, "risk": "Low"
    },
    "Mundra": {
        "id": "INMUN", "name": "Mundra", "country": "India", "region": "South Asia",
        "latitude": 22.84, "longitude": 69.70, "congestion": "Low", "dwell_time": 1.5, "vessel_count": 28, "risk": "Low"
    },
    "Chennai": {
        "id": "INMAA", "name": "Chennai", "country": "India", "region": "South Asia",
        "latitude": 13.09, "longitude": 80.30, "congestion": "Medium", "dwell_time": 3.2, "vessel_count": 21, "risk": "Low"
    },
    "Cochin": {
        "id": "INCOK", "name": "Cochin", "country": "India", "region": "South Asia",
        "latitude": 9.97, "longitude": 76.27, "congestion": "Low", "dwell_time": 2.0, "vessel_count": 15, "risk": "Low"
    },
    "Rotterdam": {
        "id": "NLRTM", "name": "Rotterdam", "country": "Netherlands", "region": "Europe",
        "latitude": 51.92, "longitude": 4.47, "congestion": "High", "dwell_time": 4.5, "vessel_count": 110, "risk": "Medium"
    },
    "Dubai": {
        "id": "AEDXB", "name": "Dubai", "country": "UAE", "region": "Middle East",
        "latitude": 25.01, "longitude": 55.06, "congestion": "Low", "dwell_time": 1.8, "vessel_count": 65, "risk": "Low"
    },
    "Shanghai": {
        "id": "CNSHA", "name": "Shanghai", "country": "China", "region": "East Asia",
        "latitude": 31.23, "longitude": 121.47, "congestion": "High", "dwell_time": 3.9, "vessel_count": 180, "risk": "Medium"
    },
    "Singapore": {
        "id": "SGSIN", "name": "Singapore", "country": "Singapore", "region": "Southeast Asia",
        "latitude": 1.35, "longitude": 103.82, "congestion": "High", "dwell_time": 2.5, "vessel_count": 145, "risk": "Low"
    },
    "Hamburg": {
        "id": "DEHAM", "name": "Hamburg", "country": "Germany", "region": "Europe",
        "latitude": 53.55, "longitude": 9.99, "congestion": "Medium", "dwell_time": 3.8, "vessel_count": 52, "risk": "Low"
    },
    "New York": {
        "id": "USNYC", "name": "New York", "country": "United States", "region": "North America",
        "latitude": 40.71, "longitude": -74.00, "congestion": "High", "dwell_time": 4.2, "vessel_count": 78, "risk": "Low"
    },
    "Kamarajar/Ennore": {
        "id": "INENR", "name": "Kamarajar/Ennore", "country": "India", "region": "South Asia",
        "latitude": 13.25, "longitude": 80.34, "congestion": "Low", "dwell_time": 2.1, "vessel_count": 12, "risk": "Low"
    },
    "Krishnapatnam": {
        "id": "INKRI", "name": "Krishnapatnam", "country": "India", "region": "South Asia",
        "latitude": 14.25, "longitude": 80.13, "congestion": "Low", "dwell_time": 2.3, "vessel_count": 10, "risk": "Low"
    },
    "Dhamra": {
        "id": "INDHR", "name": "Dhamra", "country": "India", "region": "South Asia",
        "latitude": 20.83, "longitude": 86.97, "congestion": "Low", "dwell_time": 1.9, "vessel_count": 8, "risk": "Low"
    },
    "Mormugao": {
        "id": "INMRM", "name": "Mormugao", "country": "India", "region": "South Asia",
        "latitude": 15.41, "longitude": 73.80, "congestion": "Low", "dwell_time": 2.5, "vessel_count": 11, "risk": "Low"
    },
    "Vizhinjam": {
        "id": "INVIZ", "name": "Vizhinjam", "country": "India", "region": "South Asia",
        "latitude": 8.38, "longitude": 76.99, "congestion": "Low", "dwell_time": 1.2, "vessel_count": 6, "risk": "Low"
    },
    "V.O. Chidambaranar": {
        "id": "INVOC", "name": "V.O. Chidambaranar", "country": "India", "region": "South Asia",
        "latitude": 8.75, "longitude": 78.19, "congestion": "Medium", "dwell_time": 2.9, "vessel_count": 14, "risk": "Low"
    },
    "Suez": {
        "id": "EGSUZ", "name": "Suez", "country": "Egypt", "region": "Middle East",
        "latitude": 29.97, "longitude": 32.54, "congestion": "High", "dwell_time": 5.0, "vessel_count": 95, "risk": "High"
    },
    "Port Said": {
        "id": "EGPSD", "name": "Port Said", "country": "Egypt", "region": "Middle East",
        "latitude": 31.26, "longitude": 32.30, "congestion": "High", "dwell_time": 3.5, "vessel_count": 70, "risk": "Medium"
    },
    "Salalah": {
        "id": "OMSLL", "name": "Salalah", "country": "Oman", "region": "Middle East",
        "latitude": 17.02, "longitude": 54.09, "congestion": "Low", "dwell_time": 1.8, "vessel_count": 35, "risk": "Low"
    },
    "Felixstowe": {
        "id": "GBFXT", "name": "Felixstowe", "country": "United Kingdom", "region": "Europe",
        "latitude": 51.96, "longitude": 1.35, "congestion": "High", "dwell_time": 4.1, "vessel_count": 48, "risk": "Low"
    },
    "Vancouver": {
        "id": "CAVAN", "name": "Vancouver", "country": "Canada", "region": "North America",
        "latitude": 49.28, "longitude": -123.12, "congestion": "Medium", "dwell_time": 3.6, "vessel_count": 45, "risk": "Low"
    },
    "Santos": {
        "id": "BRSSZ", "name": "Santos", "country": "Brazil", "region": "South America",
        "latitude": -23.95, "longitude": -46.33, "congestion": "High", "dwell_time": 4.0, "vessel_count": 60, "risk": "Medium"
    },
    "Melbourne": {
        "id": "AUMEL", "name": "Melbourne", "country": "Australia", "region": "Oceania",
        "latitude": -37.81, "longitude": 144.96, "congestion": "Low", "dwell_time": 2.2, "vessel_count": 25, "risk": "Low"
    },
    "Durban": {
        "id": "ZADUR", "name": "Durban", "country": "South Africa", "region": "Africa",
        "latitude": -29.86, "longitude": 31.03, "congestion": "High", "dwell_time": 6.5, "vessel_count": 38, "risk": "High"
    }
}

def get_port_list() -> List[str]:
    return list(PORTS.keys())

def get_port_details(name: str) -> Dict[str, Any]:
    return PORTS.get(name)

# =============================================================================
# 3. DATABASE SCHEMAS & UTILITIES
# =============================================================================
def get_connection():
    return sqlite3.connect(DATABASE, check_same_thread=False)

def hash_txt(t: str) -> str:
    try:
        import bcrypt
        return bcrypt.hashpw(t.encode('utf-8'), bcrypt.gensalt()).decode('utf-8')
    except Exception:
        return hashlib.sha256(t.encode('utf-8')).hexdigest()

def verify_hash(plain: str, hashed: str) -> bool:
    try:
        import bcrypt
        return bcrypt.checkpw(plain.encode('utf-8'), hashed.encode('utf-8'))
    except Exception:
        return hashlib.sha256(plain.encode('utf-8')).hexdigest() == hashed

def init_db():
    with get_connection() as conn:
        # Users table
        conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password TEXT,
            security_question TEXT,
            security_answer TEXT,
            role TEXT DEFAULT 'Logistics Manager',
            failed_attempts INTEGER DEFAULT 0,
            lock_until TIMESTAMP DEFAULT NULL,
            account_status TEXT DEFAULT 'active',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        """)

        # Migration logic: inspect columns and alter table if missing
        cursor = conn.cursor()
        cursor.execute("PRAGMA table_info(users)")
        columns = [col[1] for col in cursor.fetchall()]
        if "security_question" not in columns:
            conn.execute("ALTER TABLE users ADD COLUMN security_question TEXT")
        if "security_answer" not in columns:
            conn.execute("ALTER TABLE users ADD COLUMN security_answer TEXT")
        if "security_answer_hash" not in columns:
            conn.execute("ALTER TABLE users ADD COLUMN security_answer_hash TEXT")

        # Carriers table
        conn.execute("""
        CREATE TABLE IF NOT EXISTS carriers (
            carrier_id TEXT PRIMARY KEY,
            carrier_name TEXT,
            transport_mode TEXT,
            punctuality_rate REAL,
            avg_delay_days REAL,
            fuel_surcharge_pct REAL,
            tariff_compliance_score REAL,
            tier_rating TEXT,
            flagged INTEGER DEFAULT 0
        )
        """)

        # Quotes table
        conn.execute("""
        CREATE TABLE IF NOT EXISTS quotes (
            quote_id TEXT PRIMARY KEY,
            created_by TEXT,
            origin TEXT,
            destination TEXT,
            distance_nm REAL,
            weight_tons REAL,
            shipment_mode TEXT,
            port_congestion TEXT,
            cargo_type TEXT,
            base_cost_usd REAL,
            margin_usd REAL,
            adjustment_factor REAL,
            final_cost_usd REAL,
            delay_risk_prob REAL,
            risk_summary TEXT,
            audit_flag TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        """)

        # Shipments table
        conn.execute("""
        CREATE TABLE IF NOT EXISTS shipments (
            shipment_id TEXT PRIMARY KEY,
            quote_id TEXT,
            carrier_name TEXT,
            actual_cost REAL,
            transit_days INTEGER,
            delay_days INTEGER,
            status TEXT DEFAULT 'In Transit',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        """)

        # Merged datasets
        conn.execute("""
        CREATE TABLE IF NOT EXISTS merged_datasets (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            agent_target TEXT,
            dataset_source TEXT,
            origin TEXT,
            destination TEXT,
            distance_nm REAL,
            weight_tons REAL,
            freight_cost_usd REAL,
            shipment_mode TEXT,
            port_congestion TEXT,
            dwell_time_days REAL,
            berth_capacity INTEGER,
            weather_disruption_level REAL,
            carrier_punctuality REAL,
            fuel_surcharge_pct REAL,
            compliance_status TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        """)

        # ML Models table
        conn.execute("""
        CREATE TABLE IF NOT EXISTS ml_models (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            agent_name TEXT,
            model_name TEXT,
            r2_score REAL,
            rmse REAL,
            accuracy REAL,
            training_rows INTEGER,
            file_path TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        """)

        # Notifications table
        conn.execute("""
        CREATE TABLE IF NOT EXISTS notifications (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            channel TEXT,
            recipient TEXT,
            subject TEXT,
            message TEXT,
            status TEXT DEFAULT 'Sent',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        """)

        # Chat history
        conn.execute("""
        CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL,
            role TEXT NOT NULL,
            content TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        """)
        conn.commit()

def save_ml_metrics(agent_name: str, model_name: str, r2: float, rmse: float, acc: float, rows: int, path: str):
    with get_connection() as conn:
        conn.execute("""
        INSERT INTO ml_models (agent_name, model_name, r2_score, rmse, accuracy, training_rows, file_path)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (agent_name, model_name, r2, rmse, acc, rows, path))
        conn.commit()

def load_chat_history(username: str, limit: int = 50) -> List[Dict[str, str]]:
    with get_connection() as conn:
        rows = conn.execute(
            "SELECT role, content FROM chat_history WHERE username=? ORDER BY id DESC LIMIT ?",
            (username, limit)
        ).fetchall()
    return [{"role": r[0], "content": r[1]} for r in reversed(rows)]

def save_chat_message(username: str, role: str, content: str):
    with get_connection() as conn:
        conn.execute(
            "INSERT INTO chat_history (username, role, content) VALUES (?, ?, ?)",
            (username, role, content)
        )
        conn.commit()

def clear_chat_history(username: str):
    with get_connection() as conn:
        conn.execute("DELETE FROM chat_history WHERE username=?", (username,))
        conn.commit()

def seed_all():
    init_db()
    with get_connection() as conn:
        # Seed default Admin Account
        if ADMIN_EMAIL_ID:
            cursor = conn.cursor()
            cursor.execute("SELECT id FROM users WHERE email=?", (ADMIN_EMAIL_ID,))
            u = cursor.fetchone()
            if not u:
                cursor.execute("""
                INSERT OR IGNORE INTO users
                (username, email, password, security_question, security_answer, role, account_status)
                VALUES (?, ?, ?, ?, ?, ?, ?)
                """, ("admin", ADMIN_EMAIL_ID, hash_txt(ADMIN_PASSWORD or "admin@123"),
                      "What is your pet name?", hash_txt("admin"), "admin", "active"))
                conn.commit()

        # Seed Carriers
        if not conn.execute("SELECT count(*) FROM carriers").fetchone()[0]:
            carriers = [
                ("CAR-001", "Nhava Sheva Cargo Express", "Ocean", 0.94, 1.2, 12.5, 0.98, "Apex", 0),
                ("CAR-002", "Mundra Oceanic Shipping", "Ocean", 0.91, 1.8, 13.0, 0.96, "Apex", 0),
                ("CAR-003", "JNPT Global Logistics", "Ocean", 0.88, 2.4, 14.2, 0.92, "Standard", 0),
                ("CAR-004", "Indo-Euro Air Services", "Air", 0.99, 0.2, 18.0, 0.99, "Apex", 0),
                ("CAR-005", "Chennai Fast Freight", "Air", 0.98, 0.3, 17.5, 0.99, "Apex", 0),
                ("CAR-006", "Cochin Rail & Land Link", "Rail/Truck", 0.89, 2.1, 11.0, 0.94, "Standard", 0),
            ]
            conn.executemany("""
            INSERT INTO carriers (carrier_id, carrier_name, transport_mode, punctuality_rate,
                                 avg_delay_days, fuel_surcharge_pct, tariff_compliance_score,
                                 tier_rating, flagged)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, carriers)

        # Seed Quotes
        if not conn.execute("SELECT count(*) FROM quotes").fetchone()[0]:
            quotes = [
                ("Q-101", "admin", "Mumbai (JNPT)", "Rotterdam", 8600.0, 45.0, "Ocean", "High", "Electronics", 18500.0, 3200.0, 1.15, 24304.0, 0.96, "Moderate risk of marine squalls", "Passed", "2026-07-27 10:00:00"),
                ("Q-102", "admin", "Mundra", "Dubai", 1050.0, 12.0, "Ocean", "Low", "Consumer Goods", 3200.0, 500.0, 1.0, 3700.0, 0.05, "Clear route weather", "Passed", "2026-07-27 11:00:00"),
                ("Q-103", "admin", "Chennai", "Singapore", 1600.0, 18.0, "Air", "Medium", "Pharmaceuticals", 9200.0, 1200.0, 1.05, 10860.0, 0.12, "Normal monsoon conditions", "Passed", "2026-07-27 12:00:00"),
            ]
            conn.executemany("INSERT INTO quotes VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)", quotes)

        # Seed Shipments
        if not conn.execute("SELECT count(*) FROM shipments").fetchone()[0]:
            shipments = [
                ("SH-201", "Q-101", "Nhava Sheva Cargo Express", 24304.0, 32, 2, "Delivered", "2026-07-27 10:15:00"),
                ("SH-202", "Q-102", "Mundra Oceanic Shipping", 3700.0, 5, 0, "Delivered", "2026-07-27 11:15:00"),
                ("SH-203", "Q-103", "Indo-Euro Air Services", 10860.0, 2, 0, "In Transit", "2026-07-27 12:15:00"),
            ]
            conn.executemany("INSERT INTO shipments VALUES (?,?,?,?,?,?,?,?)", shipments)
        conn.commit()

# Run database initialization
init_db()
seed_all()

# =============================================================================
# 4. JWT & AUTH UTILITIES (BACKEND ONLY)
# =============================================================================
def decode_token(token: str) -> Dict[str, Any]:
    try:
        return jwt.decode(token, JWT_SECRET_KEY, algorithms=["HS256"])
    except Exception:
        return None

def generate_token(username: str, role: str) -> str:
    payload = {
        "username": username,
        "role": role,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=24)
    }
    return jwt.encode(payload, JWT_SECRET_KEY, algorithm="HS256")

### Step 4: Write Core Backend Services Module
Writes `backend.py` containing model managers, translation algorithms, and hybrid RAG search logic.

In [ ]:
%%writefile /content/freightquote_m4/backend.py
"""
backend.py — Computational, ML, RAG, and AI Services for FreightQuote AI.
Consolidates vector_store.py, embeddings.py, retriever.py, rag_engine.py,
llm_engine.py, predict.py, weather_service.py, translation_engine.py, and admin.py helpers.
"""
import os
import re
import glob
import json
import time
import socket
import threading
import pickle
import datetime
import requests
import joblib
import numpy as np
import pandas as pd
import torch
from typing import List, Tuple, Dict, Any

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    try:
        from langchain.text_splitter import RecursiveCharacterTextSplitter
    except ImportError:
        RecursiveCharacterTextSplitter = None

try:
    from langchain_core.documents import Document
except ImportError:
    try:
        from langchain.docstore.document import Document
    except ImportError:
        Document = None

try:
    from langchain_community.vectorstores import FAISS
except ImportError:
    try:
        from langchain.vectorstores import FAISS
    except ImportError:
        FAISS = None

try:
    from langchain_community.embeddings import HuggingFaceEmbeddings
except ImportError:
    try:
        from langchain.embeddings import HuggingFaceEmbeddings
    except ImportError:
        HuggingFaceEmbeddings = None

from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
import fitz  # PyMuPDF

import config_utils
from config_utils import get_connection, save_ml_metrics

# =============================================================================
# 1. EMBEDDING MODEL & VECTOR STORE SINGLETONS
# =============================================================================
_embedding_model = None

def get_embedding_model():
    global _embedding_model
    if _embedding_model is None:
        if HuggingFaceEmbeddings is None:
            raise RuntimeError("HuggingFaceEmbeddings package not found.")
        model_name = getattr(config_utils, "EMBEDDING_MODEL_NAME", "sentence-transformers/all-MiniLM-L6-v2")
        _embedding_model = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': 'cpu'},
            encode_kwargs={'normalize_embeddings': True}
        )
    return _embedding_model

_vectorstore = None

def get_vector_store(emb=None, rag_dir=None, faiss_dir=None):
    global _vectorstore
    if _vectorstore is not None:
        return _vectorstore

    if emb is None:
        emb = get_embedding_model()

    if rag_dir is None:
        rag_dir = getattr(config_utils, "RAG_DIR", "./rag_documents")
    if faiss_dir is None:
        faiss_dir = getattr(config_utils, "FAISS_DIR", "./faiss_index")

    os.makedirs(rag_dir, exist_ok=True)
    os.makedirs(faiss_dir, exist_ok=True)

    faiss_file = os.path.join(faiss_dir, "index.faiss")

    if FAISS is not None and os.path.exists(faiss_file):
        try:
            _vectorstore = FAISS.load_local(faiss_dir, emb, allow_dangerous_deserialization=True)
            return _vectorstore
        except Exception:
            pass

    # Generate PDFs if directory has fewer than 35 PDFs
    pdf_files = glob.glob(os.path.join(rag_dir, "*.pdf"))
    if len(pdf_files) < 35:
        generate_knowledge_base_pdfs(rag_dir)
        pdf_files = glob.glob(os.path.join(rag_dir, "*.pdf"))

    raw_docs = []
    for pdf_path in pdf_files:
        try:
            doc = fitz.open(pdf_path)
            fname = os.path.basename(pdf_path)

            cat = "Logistics"
            for kw in ["Freight", "Incoterms", "Customs", "Port", "Risk", "Warehousing", "Carrier", "Import", "Transportation", "Compliance", "Inventory", "Delay", "Costs", "Corridors", "Profiles", "Guides"]:
                if kw.lower() in fname.lower():
                    cat = kw
                    break

            for i in range(len(doc)):
                t = doc[i].get_text("text").strip()
                if t and Document is not None:
                    raw_docs.append(Document(
                        page_content=t,
                        metadata={
                            "source": fname,
                            "filepath": pdf_path,
                            "page": i + 1,
                            "total_pages": len(doc),
                            "category": cat
                        }
                    ))
            doc.close()
        except Exception as e:
            print(f"Error parsing {pdf_path}: {e}")
            pass

    if RecursiveCharacterTextSplitter is not None:
        splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
        chunks = splitter.split_documents(raw_docs)
    else:
        chunks = raw_docs

    if FAISS is not None:
        _vectorstore = FAISS.from_documents(chunks, emb)
        _vectorstore.save_local(faiss_dir)
    return _vectorstore

def generate_pdf(filename, title, category, intro, sections, table_data, bullets):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    doc = SimpleDocTemplate(filename, pagesize=letter,
                            rightMargin=54, leftMargin=54,
                            topMargin=54, bottomMargin=54)
    styles = getSampleStyleSheet()
    
    title_style = ParagraphStyle(
        'DocTitle', parent=styles['Heading1'], fontName='Helvetica-Bold',
        fontSize=20, leading=24, textColor=colors.HexColor('#1A365D'), spaceAfter=15
    )
    cat_style = ParagraphStyle(
        'DocCat', parent=styles['Normal'], fontName='Helvetica-Bold',
        fontSize=9, leading=11, textColor=colors.HexColor('#D69E2E'), spaceAfter=20
    )
    h2_style = ParagraphStyle(
        'DocH2', parent=styles['Heading2'], fontName='Helvetica-Bold',
        fontSize=12, leading=15, textColor=colors.HexColor('#2C5282'), spaceBefore=14, spaceAfter=8
    )
    body_style = ParagraphStyle(
        'DocBody', parent=styles['Normal'], fontName='Helvetica',
        fontSize=10, leading=14, textColor=colors.HexColor('#2D3748'), spaceAfter=10
    )
    bullet_style = ParagraphStyle(
        'DocBullet', parent=styles['Normal'], fontName='Helvetica',
        fontSize=10, leading=14, textColor=colors.HexColor('#2D3748'),
        leftIndent=20, firstLineIndent=-10, spaceAfter=6
    )
    
    story = []
    story.append(Paragraph(title, title_style))
    story.append(Paragraph(f"FREIGHTQUOTE AI KNOWLEDGE SYSTEM &mdash; CATEGORY: {category.upper()}", cat_style))
    story.append(Spacer(1, 10))
    story.append(Paragraph("<b>1. EXECUTIVE SUMMARY & OVERVIEW</b>", h2_style))
    story.append(Paragraph(intro, body_style))
    story.append(Spacer(1, 10))
    
    for idx, (sec_title, sec_text) in enumerate(sections, 2):
        story.append(Paragraph(f"<b>{idx}. {sec_title.upper()}</b>", h2_style))
        for p in sec_text:
            story.append(Paragraph(p, body_style))
        story.append(Spacer(1, 10))
        
    story.append(PageBreak())
    story.append(Paragraph(f"<b>{len(sections)+2}. PROCESS METRICS & COMPLIANCE TARGETS</b>", h2_style))
    story.append(Spacer(1, 8))
    
    table_content = [[
        Paragraph("<b>Operation/Component</b>", body_style),
        Paragraph("<b>Standard Parameter</b>", body_style),
        Paragraph("<b>KPI Target</b>", body_style),
        Paragraph("<b>Mitigation Rule</b>", body_style)
    ]]
    for row in table_data:
        table_content.append([
            Paragraph(row[0], body_style),
            Paragraph(row[1], body_style),
            Paragraph(row[2], body_style),
            Paragraph(row[3], body_style)
        ])
    
    t = Table(table_content, colWidths=[120, 140, 100, 140])
    t.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#E2E8F0')),
        ('ALIGN', (0,0), (-1,-1), 'LEFT'),
        ('VALIGN', (0,0), (-1,-1), 'TOP'),
        ('BOTTOMPADDING', (0,0), (-1,0), 6),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#CBD5E0')),
        ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor('#F7FAFC')]),
        ('TOPPADDING', (0,0), (-1,-1), 6),
        ('BOTTOMPADDING', (0,0), (-1,-1), 6),
    ]))
    story.append(t)
    story.append(Spacer(1, 15))
    
    story.append(Paragraph(f"<b>{len(sections)+3}. STRATEGIC RECOMMENDATIONS & KEY TAKEAWAYS</b>", h2_style))
    for b in bullets:
        story.append(Paragraph(f"&bull; {b}", bullet_style))
        
    doc.build(story)

def generate_knowledge_base_pdfs(target_dir: str):
    os.makedirs(target_dir, exist_ok=True)
    try:
        from vector_store import corpus_data
    except ImportError:
        corpus_data = []
    for idx, doc in enumerate(corpus_data, 1):
        filename = os.path.join(target_dir, doc['filename'])
        generate_pdf(
            filename=filename, title=doc['title'], category=doc['category'],
            intro=doc['intro'], sections=doc['sections'], table_data=doc['table_data'],
            bullets=doc['bullets']
        )

# =============================================================================
# 2. HYBRID RETRIEVER (FAISS + BM25)
# =============================================================================
_bm25_cache = None

def get_bm25_index(vs) -> Tuple[Any, List[Document]]:
    global _bm25_cache
    if _bm25_cache is not None:
        return _bm25_cache["bm25"], _bm25_cache["docs"]
        
    faiss_dir = getattr(config_utils, "FAISS_DIR", "./faiss_index")
    bm25_path = os.path.join(faiss_dir, "bm25_index.pkl")
    
    all_docs = []
    if hasattr(vs, "docstore") and hasattr(vs.docstore, "_dict"):
        all_docs = list(vs.docstore._dict.values())
        
    if not all_docs:
        return None, []
        
    if os.path.exists(bm25_path):
        try:
            with open(bm25_path, "rb") as f:
                cached = pickle.load(f)
                if len(cached.get("docs", [])) == len(all_docs):
                    _bm25_cache = cached
                    return cached["bm25"], cached["docs"]
        except Exception:
            pass
            
    tokenized_corpus = [doc.page_content.lower().split() for doc in all_docs]
    from rank_bm25 import BM25Okapi
    bm25 = BM25Okapi(tokenized_corpus)
    
    _bm25_cache = {"bm25": bm25, "docs": all_docs}
    try:
        with open(bm25_path, "wb") as f:
            pickle.dump(_bm25_cache, f)
    except Exception:
        pass
        
    return bm25, all_docs

def clear_bm25_cache():
    global _bm25_cache
    _bm25_cache = None
    faiss_dir = getattr(config_utils, "FAISS_DIR", "./faiss_index")
    bm25_path = os.path.join(faiss_dir, "bm25_index.pkl")
    if os.path.exists(bm25_path):
        try:
            os.remove(bm25_path)
        except Exception:
            pass

def retrieve_hybrid(query: str, k: int = 4, dense_weight: float = 0.6, sparse_weight: float = 0.4) -> List[Dict[str, Any]]:
    vs = get_vector_store()
    if vs is None:
        return []
        
    bm25, all_docs = get_bm25_index(vs)
    if bm25 is None or not all_docs:
        try:
            results = vs.similarity_search_with_score(query, k=k)
            return [{
                "doc": doc, "score": float(1.0 / (1.0 + score)),
                "dense_score": float(1.0 / (1.0 + score)), "sparse_score": 0.0
            } for doc, score in results]
        except Exception:
            return []
            
    # FAISS Dense
    try:
        dense_results = vs.similarity_search_with_score(query, k=k*2)
    except Exception:
        dense_results = []
        
    dense_sims = {}
    for doc, dist in dense_results:
        if dist > 1.3:
            continue
        dense_sims[doc.page_content] = (doc, 1.0 / (1.0 + float(dist)))
        
    # BM25 Sparse
    STOPWORDS = {"who", "was", "the", "first", "of", "is", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for", "with", "by", "what", "where", "which", "how", "why"}
    tokens = [t for t in query.lower().split() if t not in STOPWORDS and t.isalnum()]
    
    if tokens:
        bm25_scores = bm25.get_scores(tokens)
        sparse_indices = np.argsort(bm25_scores)[::-1][:k*2]
        sparse_results = []
        for idx in sparse_indices:
            score = float(bm25_scores[idx])
            if score > 0.5:
                sparse_results.append((all_docs[idx], score))
    else:
        sparse_results = []
        
    # Absolute combination
    scored_docs = {}
    for content, (doc, sim) in dense_sims.items():
        scored_docs[content] = {
            "doc": doc, "score": dense_weight * sim, "dense_score": sim, "sparse_score": 0.0
        }
        
    for doc, score in sparse_results:
        content = doc.page_content
        soft_sparse = score / (score + 10.0)
        if content in scored_docs:
            scored_docs[content]["score"] += sparse_weight * soft_sparse
            scored_docs[content]["sparse_score"] = soft_sparse
        else:
            scored_docs[content] = {
                "doc": doc, "score": sparse_weight * soft_sparse, "dense_score": 0.0, "sparse_score": soft_sparse
            }
            
    ranked = sorted(scored_docs.values(), key=lambda x: x["score"], reverse=True)[:k]
    return ranked

def retrieve_context_and_citations(query: str, k: int = 4) -> Dict[str, Any]:
    try:
        import streamlit as st
        dense_w = st.session_state.get("faiss_weight", 0.6)
        sparse_w = st.session_state.get("bm25_weight", 0.4)
    except Exception:
        dense_w = 0.6
        sparse_w = 0.4
        
    results = retrieve_hybrid(query, k=k, dense_weight=dense_w, sparse_weight=sparse_w)
    
    context_blocks = []
    citations = []
    
    for idx, item in enumerate(results, 1):
        doc = item["doc"]
        score = item["score"]
        src = doc.metadata.get("source", "Document")
        page = doc.metadata.get("page", 1)
        cat = doc.metadata.get("category", "Logistics")
        filepath = doc.metadata.get("filepath", "")
        
        chunk_id = f"fq_{src.split('_')[0]}_{page}_{idx}"
        
        context_blocks.append(f"[{src} Page {page} | Category: {cat}]: {doc.page_content}")
        citations.append({
            "source": src, "filepath": filepath, "page": page,
            "total_pages": doc.metadata.get("total_pages", 1), "category": cat,
            "score": float(score), "dense_score": float(item["dense_score"]),
            "sparse_score": float(item["sparse_score"]), "chunk_id": chunk_id,
            "snippet": doc.page_content[:150] + "...", "full_chunk": doc.page_content
        })
        
    return {
        "query": query, "context": "\n\n".join(context_blocks), "citations": citations
    }

# =============================================================================
# 3. RAG GROUNDING & Refusal
# =============================================================================
def validate_generated_answer(answer: str, context: str) -> Dict[str, Any]:
    numeric_pattern = r'\b\d+(?:,\d+)*(?:\.\d+)?%?\b'
    ans_nums = set(re.findall(numeric_pattern, answer))
    ctx_nums = set(re.findall(numeric_pattern, context))
    ans_nums = {num for num in ans_nums if len(num.replace('%', '')) > 1 or num in ["0", "1", "2"]}
    
    currencies = ["$", "€", "£", "USD", "EUR", "INR"]
    ans_curr = {c for c in currencies if c in answer}
    ctx_curr = {c for c in currencies if c in context}
    
    unsupported_nums = ans_nums - ctx_nums
    unsupported_curr = ans_curr - ctx_curr
    
    return {
        "is_valid": len(unsupported_nums) == 0 and len(unsupported_curr) == 0,
        "unsupported_numbers": list(unsupported_nums),
        "unsupported_currencies": list(unsupported_curr)
    }

def execute_rag_query(query: str, k: int = 4) -> Dict[str, Any]:
    start_time = time.time()
    
    retrieval_res = retrieve_context_and_citations(query, k=k)
    citations = retrieval_res["citations"]
    context_text = retrieval_res["context"]
    
    latency = time.time() - start_time
    top_cat = citations[0]["category"] if citations else "General Logistics"
    top_score = citations[0]["score"] if citations else 0.0
    
    if not citations or top_score < 0.35:
        grounding_level = "🔴 INSUFFICIENT"
        confidence_desc = "Insufficient verified context to answer this query."
    elif top_score < 0.60:
        grounding_level = "🟡 LIMITED"
        confidence_desc = "Limited context found. Answer may be incomplete or generic."
    else:
        grounding_level = "🟢 HIGH"
        confidence_desc = "Strong matching documentation found in the knowledge base."
        
    if grounding_level == "🔴 INSUFFICIENT":
        answer = "I don't have enough verified information in the connected database or knowledge base to answer this reliably."
        validation_res = {"is_valid": True, "unsupported_numbers": [], "unsupported_currencies": []}
    else:
        system_prompt = (
            "You are the FreightQuote AI Copilot. "
            "Your task is to answer the user's query using ONLY the provided verified context.\n"
            "Rules:\n"
            "1. Answer only using the supplied verified context. Do not use external knowledge.\n"
            "2. Do not invent or fabricate facts, numbers, or sources.\n"
            "3. Do not claim live data unless the context explicitly mentions it is live.\n"
            "4. If the context is insufficient to answer the query, explicitly say: 'I don't have enough verified information to answer this.'\n"
            "5. Distinguish calculated values from retrieved values.\n"
            "6. Use professional business language and preserve maritime terminology.\n"
            "7. Never fabricate a citation."
        )
        user_prompt = (
            f"VERIFIED CONTEXT:\n{context_text}\n\n"
            f"USER QUERY: {query}\n\n"
            f"Write a professional, grounded response based ONLY on the context above:"
        )
        answer = generate_grounded_answer(system_prompt, user_prompt, max_tokens=350)
        validation_res = validate_generated_answer(answer, context_text)
        
        if not validation_res["is_valid"]:
            grounding_level = "🟡 LIMITED"
            confidence_desc = f"Warning: Answer contains numbers {validation_res['unsupported_numbers']} not present in context."
            
    return {
        "question": query, "answer": answer, "context": context_text,
        "citations": citations, "top_category": top_cat,
        "grounding_level": grounding_level, "confidence_desc": confidence_desc,
        "validation": validation_res, "latency_sec": round(latency, 4)
    }

# =============================================================================
# 4. LLM / QWEN 2.5 ENGINE COORDINATOR
# =============================================================================
_model = None
_tokenizer = None
_load_lock = threading.Lock()
_warmup_thread_started = False

def get_model():
    global _model, _tokenizer
    if _model is not None:
        return _model, _tokenizer
    with _load_lock:
        if _model is not None:
            return _model, _tokenizer

        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

        kw = {"token": config_utils.HF_TOKEN} if config_utils.HF_TOKEN else {}
        _tokenizer = AutoTokenizer.from_pretrained(config_utils.QWEN_MODEL, **kw)
        try:
            _model = AutoModelForCausalLM.from_pretrained(
                config_utils.QWEN_MODEL, quantization_config=bnb, device_map="auto",
                torch_dtype=torch.float16, low_cpu_mem_usage=True, attn_implementation="sdpa", **kw
            )
        except Exception:
            _model = AutoModelForCausalLM.from_pretrained(
                config_utils.QWEN_MODEL, quantization_config=bnb, device_map="auto",
                torch_dtype=torch.float16, low_cpu_mem_usage=True, attn_implementation="eager", **kw
            )
        _model.eval()
    return _model, _tokenizer

def warmup_llm():
    try:
        get_model()
        return _model is not None
    except Exception:
        return False

def is_llm_loaded():
    return _model is not None

def start_background_warmup():
    global _warmup_thread_started
    if _warmup_thread_started:
        return
    _warmup_thread_started = True
    threading.Thread(target=warmup_llm, daemon=True).start()

def _run_llm(msgs, max_tokens=250, greedy=True):
    try:
        model, tok = get_model()
        tmpl = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tok(tmpl, return_tensors="pt").to(model.device)

        gen_kw = {
            "max_new_tokens": max_tokens, "use_cache": True,
            "pad_token_id": tok.eos_token_id, "eos_token_id": tok.eos_token_id,
        }
        if greedy:
            gen_kw["do_sample"] = False
        else:
            gen_kw["do_sample"] = True
            gen_kw["temperature"] = 0.2
            gen_kw["top_p"] = 0.9

        with torch.inference_mode():
            out = model.generate(**inputs, **gen_kw)
        return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    except Exception as e:
        return f"LLM error: {e}"

def orchestrate_3_agents_query(user_question, agent1_context, agent2_context, agent3_context, db_stats=None):
    if not is_llm_loaded():
        cost = agent1_context.get("base_rate_usd", 15000)
        delay_risk = agent2_context.get("delay_risk_pct", 50)
        compliance = agent3_context.get("compliance", "Passed")
        advice = f"Based on live routing data: Predicted freight cost is ${cost:,.2f} with a {delay_risk}% late arrival probability. "
        if delay_risk > 60:
            advice += "Action required: Shift cargo to alternative routing or select a carrier with higher punctuality."
        else:
            advice += f"Action: Proceed with shipment. Compliance rating [{compliance}] is satisfactory."
        return advice

    sys_p = (
        "You are the FreightQuote AI Copilot. "
        "Formulate a professional, actionable 2-sentence executive summary based on the agent reports provided."
    )
    user_p = (
        f"QUERY: {user_question}\n"
        f"AGENT 1 (Pricing): {json.dumps(agent1_context)}\n"
        f"AGENT 2 (Delay): {json.dumps(agent2_context)}\n"
        f"AGENT 3 (Compliance): {json.dumps(agent3_context)}\n"
    )
    if db_stats:
        user_p += f"SYSTEM STATUS: {json.dumps(db_stats)}"
        
    return _run_llm([
        {"role": "system", "content": sys_p},
        {"role": "user", "content": user_p}
    ], max_tokens=150)

def generate_debate_and_synthesis(user_query, agent1_context, agent2_context, agent3_context, db_stats=None):
    if not is_llm_loaded():
        cost = agent1_context.get("base_rate_usd", 15000)
        delay = agent2_context.get("delay_risk_pct", 50)
        compliance = agent3_context.get("compliance", "Passed")
        return {
            "agent1": f"Cost is estimated at ${cost:,.0f}. Congestion may trigger extra harbor dwell surcharges.",
            "agent2": f"Delay risk stands at {delay}%. Marine wind patterns and locks indicate minor schedule latency.",
            "agent3": f"Compliance status is [{compliance}]. Documentation verification completed successfully.",
            "synthesis": f"Recommended Strategy: Proceed with standard scheduling, but lock in Apex carrier space early."
        }

    sys_p = (
        "You are the FreightQuote AI Multi-Agent Coordinator. "
        "Formulate individual debates and a synthesis. Reply STRICTLY in this format:\n"
        "[AGENT 1]: <1 sentence pricing perspective>\n"
        "[AGENT 2]: <1 sentence route delay perspective>\n"
        "[AGENT 3]: <1 sentence carrier audit perspective>\n"
        "[SYNTHESIS]: <2 sentences executive synthesis>"
    )
    user_p = (
        f"QUERY: {user_query}\n"
        f"AGENT 1: {json.dumps(agent1_context)}\n"
        f"AGENT 2: {json.dumps(agent2_context)}\n"
        f"AGENT 3: {json.dumps(agent3_context)}"
    )
    raw = _run_llm([
        {"role": "system", "content": sys_p},
        {"role": "user", "content": user_p}
    ], max_tokens=200)

    res = {
        "agent1": "Pricing models indicate cost stability.",
        "agent2": "Weather vectors indicate minor delay risks.",
        "agent3": "Carrier checks suggest standard compliance levels.",
        "synthesis": raw
    }

    try:
        for key, tag, nxt in [
            ("agent1", "AGENT 1", "AGENT 2"),
            ("agent2", "AGENT 2", "AGENT 3"),
            ("agent3", "AGENT 3", "SYNTHESIS")
        ]:
            m = re.search(rf"\[{tag}\]:\s*(.*?)(?=\[{nxt}\]|\Z)", raw, re.DOTALL | re.IGNORECASE)
            if m:
                res[key] = m.group(1).strip()
        m = re.search(r"\[SYNTHESIS\]:\s*(.*)", raw, re.DOTALL | re.IGNORECASE)
        if m:
            res["synthesis"] = m.group(1).strip()
    except Exception:
        pass

    return res

def logistics_copilot(freight_cost, delay_prediction, compliance_prediction, shipment_details):
    if not is_llm_loaded():
        return f"""
### 📋 Shipment Audit & Advisory Report

- **Estimated Freight Cost**: {freight_cost}
- **Predicted Delay Risk**: {delay_prediction}
- **Carrier Compliance Status**: {compliance_prediction}
- **Cargo Specifications**: {shipment_details}

> [!NOTE]
> Ensure cargo insurance is active due to simulated delay risks.

#### Recommended Strategy
Proceed with standard booking, but secure carrier space early to mitigate dwell bottlenecks.

#### ⚙️ Structured Audit Action
```json
{{
  "action": "PROCEED",
  "audit_flag": "Passed",
  "recommended_escort": "None",
  "cost_optimization_opportunity": "None"
}}
```
"""
    sys_p = (
        "You are the FreightQuote AI Agent. Generate a formatted logistics audit report. "
        "Conclude with a valid JSON block containing keys: 'action' (PROCEED/HOLD), "
        "'audit_flag' (Passed/Flagged), 'recommended_escort' (Required/None), "
        "and 'cost_optimization_opportunity' (USD amount/None)."
    )
    user_p = (
        f"Shipment details: {shipment_details}\n"
        f"Freight Cost: {freight_cost}\n"
        f"Delay Risk: {delay_prediction}\n"
        f"Carrier Compliance: {compliance_prediction}\n"
    )
    return _run_llm([
        {"role": "system", "content": sys_p},
        {"role": "user", "content": user_p}
    ], max_tokens=350, greedy=False)

def generate_grounded_answer(system_prompt: str, user_prompt: str, max_tokens: int = 350) -> str:
    if not is_llm_loaded():
        return "Advisory Fallback: Qwen-2.5 model is currently in Standby. Please check system health or verify GPU loading status."
    
    return _run_llm([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ], max_tokens=max_tokens)

# =============================================================================
# 5. NLLB TRANSLATION & QUALITY GUARD
# =============================================================================
NLLB_LANGS = {
    "English": "eng_Latn",
    "Tamil (தமிழ்)": "tam_Taml",
    "Hindi (हिंदी)": "hin_Deva",
    "Telugu (తెలుగు)": "tel_Telu",
    "Kannada (ಕನ್ನಡ)": "kan_Knda",
    "Malayalam (മലയാളം)": "mal_Mlym",
    "Marathi (मराठी)": "mar_Deva",
    "Bengali (বাংলা)": "ben_Beng",
    "Gujarati (ગુજરાતી)": "guj_Gujr",
    "Punjabi (ਪੰਜਾਬੀ)": "pan_Guru",
    "Odia (ଓଡ଼ିଆ)": "ory_Orya",
    "Assamese (অસમীয়া)": "asm_Beng",
    "Urdu (اردو)": "urd_Arab",
    "Sanskrit (संस्कृतम्)": "san_Deva",
    "Nepali (नेपाली)": "npi_Deva",
    "Sindhi (سنڌي)": "snd_Arab",
    "Sinhala (සිංහල)": "sin_Sinh",
    "French (Français)": "fra_Latn",
    "German (Deutsch)": "deu_Latn",
    "Spanish (Español)": "spa_Latn",
    "Chinese (中文)": "zho_Hans",
    "Japanese (日本語)": "jpn_Jpan",
    "Arabic (العربية)": "arb_Arab",
}

ISO_MAP = {
    "eng_Latn": "en", "tam_Taml": "ta", "hin_Deva": "hi", "tel_Telu": "te",
    "kan_Knda": "kn", "mal_Mlym": "ml", "mar_Deva": "mr", "ben_Beng": "bn",
    "guj_Gujr": "gu", "pan_Guru": "pa", "ory_Orya": "or", "asm_Beng": "as",
    "urd_Arab": "ur", "san_Deva": "sa", "npi_Deva": "ne", "snd_Arab": "sd",
    "sin_Sinh": "si", "fra_Latn": "fr", "deu_Latn": "de", "spa_Latn": "es",
    "zho_Hans": "zh-CN", "jpn_Jpan": "ja", "arb_Arab": "ar"
}

MARITIME_GLOSSARY = ["BAF", "TEU", "HS Code", "IMO", "SOLAS", "AIS", "ETA", "ETD", "FOB", "CIF", "DAP", "DDP"]

_nllb_pipeline = None
_nllb_load_error = None
_nllb_lock = threading.Lock()

def load_nllb():
    global _nllb_pipeline, _nllb_load_error
    if _nllb_pipeline is not None:
        return _nllb_pipeline
    with _nllb_lock:
        if _nllb_pipeline is not None:
            return _nllb_pipeline
        try:
            from transformers import pipeline as hf_pipeline
            device = 0 if torch.cuda.is_available() else -1
            _nllb_pipeline = hf_pipeline(
                "translation",
                model="facebook/nllb-200-distilled-600M",
                device=device,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            )
            _nllb_load_error = None
            return _nllb_pipeline
        except Exception as e:
            _nllb_load_error = str(e)
            _nllb_pipeline = False
            return _nllb_pipeline

def is_nllb_ready():
    global _nllb_pipeline
    return callable(_nllb_pipeline)

def get_nllb_status():
    if callable(_nllb_pipeline):
        return "✅ NLLB-200 Active"
    if _nllb_pipeline is False:
        return f"⚠️ NLLB-200 unavailable ({_nllb_load_error}) — using fallback translator"
    return "⏳ NLLB-200 not loaded yet"

def is_backend_port_open(port=8000):
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(0.05)
            return s.connect_ex(('127.0.0.1', port)) == 0
    except Exception:
        return False

def detect_language(text):
    if not text: return "eng_Latn"
    for ch in text:
        if '\u0b80' <= ch <= '\u0bff': return "tam_Taml"
        if '\u0900' <= ch <= '\u097f': return "hin_Deva"
        if '\u0c00' <= ch <= '\u0c7f': return "tel_Telu"
        if '\u0c80' <= ch <= '\u0cff': return "kan_Knda"
        if '\u0d00' <= ch <= '\u0d7f': return "mal_Mlym"
        if '\u0980' <= ch <= '\u09ff': return "ben_Beng"
        if '\u0a80' <= ch <= '\u0aff': return "guj_Gujr"
        if '\u0a00' <= ch <= '\u0a7f': return "pan_Guru"
        if '\u0b00' <= ch <= '\u0b7f': return "ory_Orya"
        if '\u0600' <= ch <= '\u06ff': return "arb_Arab"
        if '\u3040' <= ch <= '\u30ff' or '\u4e00' <= ch <= '\u9fff': return "jpn_Jpan"
    return "eng_Latn"

def resolve_flores_code(lang_str):
    if not lang_str: return "eng_Latn"
    if lang_str in NLLB_LANGS.values(): return lang_str
    if lang_str in NLLB_LANGS: return NLLB_LANGS[lang_str]
    for name, code in NLLB_LANGS.items():
        if lang_str.lower() in name.lower() or name.lower() in lang_str.lower():
            return code
    return "eng_Latn"

def split_into_chunks(text, max_chars=800):
    paragraphs = text.split("\n\n")
    chunks = []
    current_chunk = []
    current_len = 0
    for p in paragraphs:
        if current_len + len(p) + 2 > max_chars:
            if current_chunk:
                chunks.append("\n\n".join(current_chunk))
                current_chunk = []
                current_len = 0
            if len(p) > max_chars:
                sentences = re.split(r'(?<=[.!?]) +', p)
                for s in sentences:
                    if current_len + len(s) + 1 > max_chars:
                        if current_chunk:
                            chunks.append(" ".join(current_chunk))
                            current_chunk = []
                            current_len = 0
                        chunks.append(s)
                    else:
                        current_chunk.append(s)
                        current_len += len(s) + 1
            else:
                current_chunk.append(p)
                current_len = len(p)
        else:
            current_chunk.append(p)
            current_len += len(p) + 2
    if current_chunk:
        chunks.append("\n\n".join(current_chunk))
    return chunks

def validate_translation(original, translated, target_lang_code) -> Dict[str, Any]:
    orig_nums = set(re.findall(r'\b\d+\b', original))
    trans_nums = set(re.findall(r'\b\d+\b', translated))
    nums_preserved = orig_nums.issubset(trans_nums)
    
    currencies = ["$", "€", "£", "¥", "USD", "EUR", "INR"]
    orig_curr = [c for c in currencies if c in original]
    curr_preserved = all(c in translated for c in orig_curr)
    
    units = ["nm", "tons", "CBM", "kg", "°C", "km/h", "days"]
    orig_units = [u for u in units if re.search(r'\b' + re.escape(u) + r'\b', original)]
    units_preserved = all(re.search(re.escape(u), translated) for u in orig_units)
    
    orig_terms = [t for t in MARITIME_GLOSSARY if re.search(r'\b' + re.escape(t) + r'\b', original, re.IGNORECASE)]
    preserved_terms = [t for t in orig_terms if re.search(re.escape(t), translated, re.IGNORECASE)]
    
    terms_ratio = f"{len(preserved_terms)}/{len(orig_terms)}" if orig_terms else "N/A"
    terms_ok = len(preserved_terms) == len(orig_terms)
    len_ok = len(translated.strip()) > 0 and (len(translated) >= len(original) * 0.3)
    engine = "NLLB-200" if is_nllb_ready() else "Google Translator (Fallback)"
    is_valid = nums_preserved and curr_preserved and units_preserved and terms_ok and len_ok
    status = "Reliable" if is_valid else "Unreliable"
    
    return {
        "is_valid": is_valid,
        "language": [k for k, v in NLLB_LANGS.items() if v == target_lang_code][0] if target_lang_code in NLLB_LANGS.values() else "Unknown",
        "engine": engine,
        "terms_preserved": terms_ratio,
        "numbers_preserved": nums_preserved,
        "currency_preserved": curr_preserved,
        "units_preserved": units_preserved,
        "status": status
    }

def _translate_uncached(text, src_lang="eng_Latn", tgt_lang="eng_Latn"):
    if not text or str(text).strip() == "": return text, None
    s_code = resolve_flores_code(src_lang)
    t_code = resolve_flores_code(tgt_lang)
    if s_code == t_code: return text, None
    last_err = None
    try:
        pipe = load_nllb()
        if callable(pipe):
            res = pipe(text, src_lang=s_code, tgt_lang=t_code)
            if res and len(res) > 0:
                out = res[0].get("translation_text", "")
                if out and out.strip(): return out, None
            last_err = "nllb: empty result"
        else: last_err = f"nllb: model not loaded ({_nllb_load_error})"
    except Exception as e: last_err = f"nllb: {e}"

    if is_backend_port_open(8000):
        try:
            res = requests.post("http://localhost:8000/translate", json={"text": text, "src_lang": s_code, "tgt_lang": t_code}, timeout=3)
            if res.status_code == 200:
                ans = res.json().get("result", "")
                if ans and ans != text: return ans, None
        except Exception as e: last_err = f"backend: {e}"

    try:
        from deep_translator import GoogleTranslator
        source_iso = ISO_MAP.get(s_code, "auto")
        target_iso = ISO_MAP.get(t_code, "en")
        if source_iso != target_iso:
            translated = GoogleTranslator(source=source_iso if source_iso != "en" or s_code == "eng_Latn" else "auto", target=target_iso).translate(text)
            if translated and translated.strip(): return translated, None
            last_err = "deep_translator: empty result"
    except Exception as e: last_err = f"deep_translator: {e}"
    return text, last_err or "no translation backend available"

def translate_text(text, src_lang="eng_Latn", tgt_lang="eng_Latn", target_lang=None) -> str:
    import streamlit as st
    if target_lang: tgt_lang = target_lang
    if not text or str(text).strip() == "": return text
    s_code = resolve_flores_code(src_lang)
    t_code = resolve_flores_code(tgt_lang)
    if s_code == t_code: return text
    chunks = split_into_chunks(text, max_chars=800)
    translated_chunks = []
    for chunk in chunks:
        trans_chunk, err = _translate_uncached(chunk, src_lang=s_code, tgt_lang=t_code)
        translated_chunks.append(trans_chunk)
    final_translation = "\n\n".join(translated_chunks)
    validation = validate_translation(text, final_translation, t_code)
    st.session_state["translation_quality"] = validation
    if not validation["is_valid"]: return text
    return final_translation

# =============================================================================
# 6. LIVE WEATHER SERVICE (OPEN-METEO)
# =============================================================================
WEATHER_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Depositing rime fog", 51: "Light drizzle", 53: "Moderate drizzle",
    55: "Dense drizzle", 56: "Light freezing drizzle", 57: "Dense freezing drizzle",
    61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain", 66: "Light freezing rain",
    67: "Heavy freezing rain", 71: "Slight snow fall", 73: "Moderate snow fall",
    75: "Heavy snow fall", 77: "Snow grains", 80: "Slight rain showers",
    81: "Moderate rain showers", 82: "Violent rain showers", 85: "Slight snow showers",
    86: "Heavy snow showers", 95: "Thunderstorm", 96: "Thunderstorm with slight hail",
    99: "Thunderstorm with heavy hail"
}

def get_live_weather(latitude: float, longitude: float) -> Dict[str, Any]:
    url = "https://api.open-meteo.com/v1/forecast"
    params = {"latitude": latitude, "longitude": longitude, "current_weather": "true", "timezone": "auto"}
    try:
        response = requests.get(url, params=params, timeout=5)
        response.raise_for_status()
        data = response.json()
        current = data.get("current_weather", {})
        if not current: return _get_fallback_weather("No current weather section in API response.")
        temp = current.get("temperature")
        wind = current.get("windspeed")
        code = int(current.get("weathercode", 0))
        desc = WEATHER_CODES.get(code, f"Unknown Code {code}")
        risk = "Low"
        risk_score = 0.1
        if wind > 50 or code in [65, 67, 82, 95, 96, 99]:
            risk = "High"
            risk_score = 0.8
        elif wind > 25 or code in [55, 57, 63, 73, 75, 81, 86, 45, 48]:
            risk = "Medium"
            risk_score = 0.5
        return {
            "status": "success", "source": "🟢 LIVE API DATA", "temperature_c": temp,
            "wind_speed_kmh": wind, "weather_code": code, "condition": desc,
            "risk_level": risk, "risk_score": risk_score,
            "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
    except Exception as e:
        return _get_fallback_weather(str(e))

def _get_fallback_weather(reason: str) -> Dict[str, Any]:
    return {
        "status": "fallback", "source": "⚪ SYNTHETIC / DEMO DATA (API Offline: " + reason[:30] + ")",
        "temperature_c": 26.5, "wind_speed_kmh": 12.0, "weather_code": 1,
        "condition": "Mainly clear (Demo)", "risk_level": "Low", "risk_score": 0.2,
        "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

# =============================================================================
# 7. MACHINE LEARNING PREDICTION MODELLER
# =============================================================================
_models = {"agent1": None, "agent2": None, "agent3": None}

def load_models_if_needed():
    global _models
    if _models["agent1"] is None and os.path.exists(config_utils.AGENT1_MODEL_PATH):
        try: _models["agent1"] = joblib.load(config_utils.AGENT1_MODEL_PATH)
        except Exception: pass
    if _models["agent2"] is None and os.path.exists(config_utils.AGENT2_MODEL_PATH):
        try: _models["agent2"] = joblib.load(config_utils.AGENT2_MODEL_PATH)
        except Exception: pass
    if _models["agent3"] is None and os.path.exists(config_utils.AGENT3_MODEL_PATH):
        try: _models["agent3"] = joblib.load(config_utils.AGENT3_MODEL_PATH)
        except Exception: pass

def get_model_status() -> Dict[str, bool]:
    load_models_if_needed()
    return {
        "Freight Model (Agent 1)": _models["agent1"] is not None,
        "Delay Model (Agent 2)": _models["agent2"] is not None,
        "Compliance Model (Agent 3)": _models["agent3"] is not None
    }

def calculate_confidence_band(model, row_data, is_regressor=False) -> Tuple[float, float, float]:
    if model is None:
        if is_regressor:
            dist, weight, cong, fuel, cargo, dwell = row_data
            base = (dist * 1.8 + weight * 48.0 + cong * 1500) * fuel
            return float(base), float(base * 0.90), float(base * 1.10)
        else:
            return 0.5, 0.42, 0.58

    if is_regressor:
        if hasattr(model, "estimators_") and not isinstance(model.estimators_[0], np.ndarray):
            try:
                preds = [t.predict([row_data])[0] for t in model.estimators_]
                mean_p = float(np.mean(preds))
                std_p = float(np.std(preds))
                if std_p == 0:
                    std_p = mean_p * 0.05
                return mean_p, mean_p - 1.96 * std_p, mean_p + 1.96 * std_p
            except Exception:
                pass
        pred = float(model.predict([row_data])[0])
        std_p = pred * 0.05
        return pred, pred - 1.96 * std_p, pred + 1.96 * std_p
    else:
        if hasattr(model, "predict_proba"):
            prob = float(model.predict_proba([row_data])[0][1])
        else:
            prob = float(np.clip(model.predict([row_data])[0], 0, 1))

        n, z = 300, 1.96
        denom = 1.0 + z**2 / n
        center = (prob + z**2 / (2 * n)) / denom
        spread = z * np.sqrt((prob * (1.0 - prob) + z**2 / (4 * n)) / n) / denom
        lo = max(0.0, center - spread)
        hi = min(1.0, center + spread)
        return prob, lo, hi

def predict_freight(dist, weight, cong_v, fuel, cargo_v, dwell) -> Tuple[float, float, float]:
    load_models_if_needed()
    row = [dist, weight, cong_v, fuel, cargo_v, dwell]
    return calculate_confidence_band(_models["agent1"], row, is_regressor=True)

def get_quote_breakdown(mean_cost: float, dist: float, weight: float, cong_v: int, fuel: float, cargo_v: int, dwell: float) -> dict:
    baf = mean_cost * 0.12 * fuel
    customs = mean_cost * 0.08 * (1.25 if cargo_v == 2 else (1.10 if cargo_v == 1 else 1.0))
    terminal = mean_cost * 0.06 * (1.40 if cong_v == 2 else (1.15 if cong_v == 1 else 1.0))
    insurance = mean_cost * 0.04 * (2.00 if cargo_v == 2 else (1.50 if cargo_v == 1 else 1.0))
    base_freight = mean_cost - (baf + customs + terminal + insurance)
    if base_freight < 0:
        base_freight = mean_cost * 0.70
        baf = mean_cost * 0.12
        customs = mean_cost * 0.08
        terminal = mean_cost * 0.06
        insurance = mean_cost * 0.04
    drivers = {"BAF (Fuel Surcharge)": baf, "Customs Fees": customs, "Terminal Handling": terminal, "Marine Insurance": insurance}
    main_driver = max(drivers, key=drivers.get)
    return {"Base Freight": round(base_freight, 2), "BAF": round(baf, 2), "Customs": round(customs, 2), "Terminal": round(terminal, 2), "Insurance": round(insurance, 2), "Total": round(mean_cost, 2), "Main Driver": main_driver}

def predict_delay(dwell, berth, route_length, weather, canal, season_risk) -> Tuple[float, float, float]:
    load_models_if_needed()
    row = [dwell, berth, route_length, weather, canal, season_risk]
    return calculate_confidence_band(_models["agent2"], row, is_regressor=False)

def predict_compliance(punct, avg_delay, complaint, fuel_sc, tariff, docs) -> Tuple[float, float, float]:
    load_models_if_needed()
    row = [punct, avg_delay, complaint, fuel_sc, tariff, docs]
    return calculate_confidence_band(_models["agent3"], row, is_regressor=False)

# =============================================================================
# 8. ADMIN DATABASE LIFECYCLE HELPERS
# =============================================================================
def get_all_users(search_query=""):
    with get_connection() as conn:
        cursor = conn.cursor()
        if search_query:
            cursor.execute("""
            SELECT id, username, email, role, failed_attempts, account_status
            FROM users WHERE username LIKE ? OR email LIKE ?
            """, (f"%{search_query}%", f"%{search_query}%"))
        else:
            cursor.execute("SELECT id, username, email, role, failed_attempts, account_status FROM users")
        return cursor.fetchall()

def delete_user(user_id):
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM users WHERE id=?", (user_id,))
        conn.commit()

def unlock_user(user_id):
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("UPDATE users SET failed_attempts=0, lock_until=NULL, account_status='active' WHERE id=?", (user_id,))
        conn.commit()

def update_role(user_id, role):
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("UPDATE users SET role=? WHERE id=?", (role, user_id))
        conn.commit()

def add_user(username, email, password, role):
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("""
        INSERT INTO users (username, email, password, role, account_status)
        VALUES (?, ?, ?, ?, 'active')
        """, (username, email, config_utils.hash_txt(password), role))
        conn.commit()

def get_latest_model_metrics(agent_name):
    with get_connection() as conn:
        cursor = conn.cursor()
        try:
            cursor.execute("""
            SELECT model_name, r2_score, rmse, accuracy, training_rows, created_at
            FROM ml_models WHERE agent_name=?
            ORDER BY id DESC LIMIT 1
            """, (agent_name,))
            return cursor.fetchone()
        except Exception:
            return None


### Step 5: Write Streamlit UI Orchestrator File
Writes `app.py` containing the main navigation tabs, dashboard tiles, pricing widgets, comparison views, and system health matrix.

In [ ]:
%%writefile /content/freightquote_m4/app.py
"""
app.py — Main Streamlit application orchestrator for FreightQuote AI.
Consolidates the entire UI layer, styles, auth panels, and dashboard views.
Imports business and AI logic from backend.py and config_utils.py.
"""
import os
import sqlite3
import datetime
import time
import re
import numpy as np
import pandas as pd
import streamlit as st
from streamlit_option_menu import option_menu
import plotly.graph_objects as go
import jwt

# Consolidated Configurations & Backends
import config_utils
import backend

# =============================================================================
# 1. THEME STYLING & CYBERNETIC LOOK (From ui_theme.py)
# =============================================================================
COLORS = {
    "bg_main":       "#05070D",       # Background (Dark Metallic Blue-Grey)
    "bg_card":       "#151A24",       # Cards (Dark Gunmetal Card Background)
    "bg_alt":        "#0E111A",       # Secondary Background
    "border":        "#2E3440",       # Gunmetal Border
    "border_gold":   "#D4AF37",       # Metallic Gold
    "border_red":    "#8A000A",       # Deep Red
    "accent":        "#D4AF37",       # Metallic Gold
    "accent_orange": "#B00012",       # Metallic Crimson
    "accent_subtle": "#A9711D",       # Bronze Gold
    "accent_text":   "#05070D",       # Dark text on gold
    "text_heading":  "#FFFFFF",       # Text White
    "text_body":     "#D5D5D5",       # Secondary Text
    "text_main":     "#D5D5D5",
    "text_muted":    "#9CA3AF",       # Muted Gray
    "accent_gold":   "#F4C430",       # Bright Gold
    "accent_red":    "#B00012",       # Metallic Crimson
    "accent_wine":   "#5B0006",       # Dark Wine
    "accent_arc":    "#48C9FF",       # Arc Reactor Blue
    "gunmetal":      "#2E3440",       # Gunmetal
    "green":         "#3CB371",       # Success Green
    "yellow":        "#F4C430",       # Warning Gold
    "red":           "#E63946",       # Danger Red
    "success":       "#3CB371",
    "danger":        "#E63946",
}

CYBER_CSS = f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700&family=Space+Grotesk:wght@600;700&family=Fira+Code:wght@500;700&display=swap');

html, body, [class*="css"] {{
    font-family: 'Plus Jakarta Sans', sans-serif;
    color: {COLORS["text_body"]};
    background-color: {COLORS["bg_main"]};
    background-image: radial-gradient(circle at 50% 10%, #0f1524 0%, {COLORS["bg_main"]} 70%);
}}

h1, h2, h3, h4, h5, h6 {{
    font-family: 'Space Grotesk', sans-serif;
    color: {COLORS["text_heading"]};
    font-weight: 700;
    text-transform: uppercase;
    letter-spacing: 0.5px;
}}

.pn-card {{
    background: linear-gradient(135deg, {COLORS["bg_card"]} 0%, #1a2230 100%);
    border: 1px solid {COLORS["accent"]};
    border-radius: 8px;
    padding: 24px;
    margin-bottom: 24px;
    box-shadow: 0 4px 20px rgba(0, 0, 0, 0.4), inset 0 1px 0 rgba(255, 255, 255, 0.05);
    transition: all 0.3s cubic-bezier(0.25, 0.8, 0.25, 1);
}}
.pn-card:hover {{
    border-color: {COLORS["accent_gold"]};
    transform: translateY(-3px);
    box-shadow: 0 12px 30px rgba(0, 0, 0, 0.6), 0 0 15px rgba(244, 196, 48, 0.25);
}}

.pn-card-red {{
    background: linear-gradient(135deg, {COLORS["bg_card"]} 0%, #1a2230 100%);
    border: 1px solid {COLORS["border_red"]};
    border-radius: 8px;
    padding: 24px;
    margin-bottom: 24px;
    box-shadow: 0 4px 20px rgba(0, 0, 0, 0.4);
    transition: all 0.3s ease;
}}
.pn-card-red:hover {{
    border-color: {COLORS["accent_red"]};
    box-shadow: 0 10px 25px rgba(0, 0, 0, 0.5), 0 0 12px rgba(176, 0, 18, 0.3);
}}

.pn-card-alt {{
    background: {COLORS["bg_alt"]};
    border: 1px solid {COLORS["gunmetal"]};
    border-radius: 8px;
    padding: 20px;
    margin-bottom: 20px;
}}

.pn-badge {{
    display: inline-block;
    padding: 4px 14px;
    border: 1px solid {COLORS["gunmetal"]};
    border-radius: 4px;
    font-family: 'Fira Code', monospace;
    font-weight: 700;
    font-size: 11px;
    background: #090d16;
    color: {COLORS["text_heading"]};
}}

.status-ready {{
    background-color: rgba(60, 179, 113, 0.15) !important;
    color: {COLORS["success"]} !important;
    border-color: {COLORS["success"]} !important;
}}
.status-running {{
    background-color: rgba(244, 196, 48, 0.15) !important;
    color: {COLORS["accent_gold"]} !important;
    border-color: {COLORS["accent_gold"]} !important;
}}
.status-offline {{
    background-color: rgba(230, 57, 70, 0.15) !important;
    color: {COLORS["danger"]} !important;
    border-color: {COLORS["danger"]} !important;
}}

.agent-badge {{
    display: inline-block;
    padding: 4px 14px;
    background: linear-gradient(135deg, {COLORS["accent_gold"]} 0%, #c88a00 100%);
    color: #020617;
    border-radius: 4px;
    font-family: 'Space Grotesk', sans-serif;
    font-weight: 700;
    font-size: 12px;
    text-transform: uppercase;
    box-shadow: 0 0 8px rgba(244, 196, 48, 0.3);
}}

.arc-accent {{
    color: {COLORS["accent_arc"]};
    text-shadow: 0 0 8px rgba(72, 201, 255, 0.5);
}}

div.stButton > button {{
    background: linear-gradient(180deg, {COLORS["accent_gold"]} 0%, #c88a00 100%) !important;
    color: #020617 !important;
    font-family: 'Space Grotesk', sans-serif !important;
    font-weight: 700 !important;
    border: 1px solid {COLORS["border_gold"]} !important;
    border-radius: 24px !important;
    padding: 10px 24px !important;
    text-transform: uppercase;
    letter-spacing: 0.5px;
    transition: all 0.2s cubic-bezier(0.25, 0.8, 0.25, 1) !important;
    box-shadow: 0 4px 10px rgba(0, 0, 0, 0.3) !important;
}}
div.stButton > button:hover {{
    transform: scale(1.02) !important;
    box-shadow: 0 0 15px rgba(244, 196, 48, 0.5) !important;
    border-color: #ffffff !important;
}}
div.stButton > button:active {{
    transform: scale(0.98) !important;
}}

.red-btn-container button {{
    background: linear-gradient(180deg, {COLORS["accent_red"]} 0%, #7a0008 100%) !important;
    border-color: {COLORS["border_red"]} !important;
    color: #FFFFFF !important;
    box-shadow: 0 4px 10px rgba(0, 0, 0, 0.3) !important;
}}
.red-btn-container button:hover {{
    box-shadow: 0 0 15px rgba(176, 0, 18, 0.5) !important;
    border-color: #ffffff !important;
}}

div[data-baseweb="input"] > div, div[data-baseweb="select"] > div {{
    background-color: {COLORS["bg_alt"]} !important;
    border: 1px solid {COLORS["gunmetal"]} !important;
    border-radius: 6px !important;
    color: {COLORS["text_heading"]} !important;
    box-shadow: inset 0 2px 4px rgba(0,0,0,0.5) !important;
    transition: all 0.2s ease;
}}
div[data-baseweb="input"] > div:focus-within, div[data-baseweb="select"] > div:focus-within {{
    border-color: {COLORS["accent_gold"]} !important;
    box-shadow: 0 0 8px rgba(244, 196, 48, 0.3), inset 0 2px 4px rgba(0,0,0,0.5) !important;
}}

section[data-testid="stSidebar"] {{
    background-color: {COLORS["bg_alt"]} !important;
    border-right: 1px solid {COLORS["border_gold"]} !important;
}}
section[data-testid="stSidebar"] .stMarkdown {{
    color: {COLORS["text_body"]};
}}

button[data-baseweb="tab"] {{
    font-family: 'Space Grotesk', sans-serif !important;
    font-weight: 600 !important;
    color: {COLORS["text_muted"]} !important;
    background-color: transparent !important;
    border: none !important;
    padding: 12px 20px !important;
}}
button[data-baseweb="tab"][aria-selected="true"] {{
    color: {COLORS["accent_gold"]} !important;
    border-bottom: 2px solid {COLORS["accent_gold"]} !important;
    text-shadow: 0 0 8px rgba(244, 196, 48, 0.4);
}}

.top-nav-bar {{
    display: flex;
    justify-content: center;
    gap: 30px;
    background-color: {COLORS["bg_alt"]};
    border: 1px solid {COLORS["border_gold"]};
    border-radius: 8px;
    padding: 12px 24px;
    margin-bottom: 30px;
    box-shadow: 0 4px 10px rgba(0,0,0,0.3);
}}
.top-nav-item {{
    color: {COLORS["text_muted"]};
    font-family: 'Space Grotesk', sans-serif;
    font-weight: 700;
    text-decoration: none;
    font-size: 14px;
    text-transform: uppercase;
    letter-spacing: 0.5px;
    padding: 6px 16px;
    border-radius: 4px;
    transition: all 0.3s ease;
}}
.top-nav-item:hover {{
    color: {COLORS["accent_gold"]};
    background-color: rgba(244, 196, 48, 0.05);
}}
.top-nav-item-active {{
    color: {COLORS["accent_gold"]} !important;
    background-color: rgba(244, 196, 48, 0.1) !important;
    border: 1px solid {COLORS["border_gold"]};
    text-shadow: 0 0 8px rgba(244, 196, 48, 0.4);
}}
</style>
"""

def apply_theme():
    st.markdown(CYBER_CSS, unsafe_allow_html=True)

def render_header(title, subtitle="", icon="⚡"):
    st.markdown(f"""
    <div style="background:{COLORS['bg_card']}; border: 1px solid {COLORS['accent']}; border-radius:8px; padding:24px 30px; margin-bottom:30px; box-shadow: 0 8px 24px rgba(0,0,0,0.4);">
        <div style="display:flex; align-items:center; gap:20px;">
            <div class="arc-accent" style="font-size:46px; line-height:1;">{icon}</div>
            <div>
                <h1 style="margin:0; font-size:28px; letter-spacing:-0.5px; color:{COLORS['text_heading']}; font-family:'Space Grotesk',sans-serif;">{title}</h1>
                <p style="margin:6px 0 0; color:{COLORS['text_muted']}; font-size:14px; font-weight:500;">{subtitle}</p>
            </div>
        </div>
    </div>
    """, unsafe_allow_html=True)

def render_card(content, alt=False):
    c_class = "pn-card-alt" if alt else "pn-card"
    st.markdown(f'<div class="{c_class}">{content}</div>', unsafe_allow_html=True)

def risk_badge(text, level="Low"):
    color_map = {
        "Low": COLORS["success"],
        "Medium": COLORS["accent_gold"],
        "High": COLORS["danger"],
        "Critical": COLORS["danger"]
    }
    c = color_map.get(level, COLORS["accent_gold"])
    return f'<span class="pn-badge" style="background:rgba(255,255,255,0.02); color:{c}; border-color:{c}; font-weight:bold;">{text}</span>'

# =============================================================================
# 2. LOGIN PORTAL UI RENDERING (From auth.py)
# =============================================================================
def render_auth_portal():
    apply_theme()
    st.markdown(
        f'<div style="text-align:center;padding-top:40px;margin-bottom:20px;">'
        f'<h1 style="color:{COLORS["accent"]};font-size:36px;letter-spacing:1px;font-family:\'Space Grotesk\',sans-serif;">⚡ FREIGHTQUOTE AI</h1>'
        f'<p style="color:{COLORS["text_muted"]};font-size:14px;">Enterprise Maritime Freight Intelligence & Route Optimization Platform</p>'
        f'</div>',
        unsafe_allow_html=True
    )
    
    auth_mode = st.radio("Access Mode", ["Login Portal", "Register Account", "Forgot Password / OTP Recovery"], horizontal=True, label_visibility="collapsed")
    
    if auth_mode == "Login Portal":
        st.subheader("🔒 User Sign-In Gateway")
        with st.form("login_form"):
            in_user = st.text_input("Username").strip()
            in_pass = st.text_input("Password", type="password")
            btn = st.form_submit_button("Authorize & Access System", use_container_width=True)
            
            if btn:
                if not in_user or not in_pass:
                    st.error("Fields cannot be left blank.")
                else:
                    with config_utils.get_connection() as conn:
                        cursor = conn.cursor()
                        cursor.execute("SELECT id, password, role, account_status, failed_attempts FROM users WHERE username=?", (in_user,))
                        row = cursor.fetchone()
                        if row:
                            u_id, hashed_pw, role, status, attempts = row
                            if status == "locked":
                                st.error("❌ Account is locked due to security policy constraints. Contact Administrator.")
                            elif config_utils.verify_hash(in_pass, hashed_pw):
                                # Reset failed attempts
                                cursor.execute("UPDATE users SET failed_attempts=0 WHERE id=?", (u_id,))
                                conn.commit()
                                # Generate JWT Token
                                tok = config_utils.generate_token(in_user, role)
                                st.session_state["token"] = tok
                                st.session_state["username"] = in_user
                                st.session_state["role"] = role
                                st.success("Access Granted! Loading secure modules...")
                                time.sleep(0.5)
                                st.rerun()
                            else:
                                attempts += 1
                                if attempts >= 3:
                                    cursor.execute("UPDATE users SET account_status='locked', failed_attempts=? WHERE id=?", (attempts, u_id))
                                    st.error("❌ Too many failed attempts. Account has been locked.")
                                else:
                                    cursor.execute("UPDATE users SET failed_attempts=? WHERE id=?", (attempts, u_id))
                                    st.error(f"Invalid Password. Attempts remaining: {3 - attempts}")
                                conn.commit()
                        else:
                            st.error("Account username not recognized in system registry.")
                            
    elif auth_mode == "Register Account":
        st.subheader("➕ Create Logistics Account")
        with st.form("reg_form"):
            new_u = st.text_input("Username").strip()
            new_e = st.text_input("Email ID").strip()
            new_p = st.text_input("Secure Password", type="password")
            sec_q = st.selectbox("Security Question", [
                "What is your pet name?",
                "What is your first school?",
                "What is your birth city?"
            ])
            sec_a = st.text_input("Security Answer").strip()
            btn = st.form_submit_button("Register Enterprise Account", use_container_width=True)
            
            if btn:
                if not (new_u and new_e and new_p and sec_a):
                    st.error("Please fill out all input parameters.")
                else:
                    try:
                        with config_utils.get_connection() as conn:
                            conn.execute("""
                            INSERT INTO users (username, email, password, security_question, security_answer, security_answer_hash)
                            VALUES (?, ?, ?, ?, ?, ?)
                            """, (new_u, new_e, config_utils.hash_txt(new_p), sec_q, sec_a, config_utils.hash_txt(sec_a.lower())))
                            conn.commit()
                        st.success("Account successfully created! Please sign in.")
                    except Exception:
                        st.error("Username or Email already registered in the registry.")
                        
    elif auth_mode == "Forgot Password / OTP Recovery":
        st.subheader("🔑 Security Recovery Console")
        step = st.radio("Recovery Step", ["1. Verify Account Info", "2. Reset Password"], horizontal=True)
        
        if step == "1. Verify Account Info":
            with st.form("forgot_step1"):
                rec_u = st.text_input("Username").strip()
                rec_e = st.text_input("Email ID").strip()
                btn1 = st.form_submit_button("Initiate Recovery Protocol", use_container_width=True)
                
                if btn1:
                    with config_utils.get_connection() as conn:
                        u = conn.execute("SELECT security_question FROM users WHERE username=? AND email=?", (rec_u, rec_e)).fetchone()
                        if u:
                            st.session_state["rec_username"] = rec_u
                            st.session_state["rec_question"] = u[0]
                            st.success(f"Identity Verified! Question: {u[0]}")
                        else:
                            st.error("No account matching details found.")
                            
        elif step == "2. Reset Password":
            rec_username = st.session_state.get("rec_username")
            rec_q = st.session_state.get("rec_question")
            
            if not rec_username:
                st.warning("Please complete Step 1 first.")
            else:
                st.markdown(f"**Security Question:** {rec_q}")
                with st.form("forgot_step2"):
                    ans_input = st.text_input("Security Answer").strip()
                    new_pass_val = st.text_input("New Password", type="password")
                    btn2 = st.form_submit_button("Update Password", use_container_width=True)
                    
                    if btn2:
                        with config_utils.get_connection() as conn:
                            u_data = conn.execute("SELECT security_answer, security_answer_hash FROM users WHERE username=?", (rec_username,)).fetchone()
                            if u_data:
                                stored_ans, stored_hash = u_data
                                if ans_input.lower() == stored_ans.lower() or config_utils.verify_hash(ans_input.lower(), stored_hash or ""):
                                    conn.execute("UPDATE users SET password=?, failed_attempts=0, account_status='active' WHERE username=?", (config_utils.hash_txt(new_pass_val), rec_username))
                                    conn.commit()
                                    st.success("Password updated successfully! Return to Login.")
                                    st.session_state["rec_username"] = None
                                else:
                                    st.error("Incorrect security answer.")

# =============================================================================
# 3. ADMIN MANAGEMENT UI PANELS (From admin.py)
# =============================================================================
def render_user_management():
    st.markdown('<h2 style="margin-top:0;color:white;">👥 User Management Console</h2>', unsafe_allow_html=True)
    render_card(
        '<h3>👥 Enterprise Access Control</h3>'
        '<p style="color:#94a3b8;font-size:13px;margin:0;">Manage user accounts, roles, lockouts, and administrative credentials.</p>'
    )
    
    with config_utils.get_connection() as conn:
        tot_users = conn.execute("SELECT count(*) FROM users").fetchone()[0]
        tot_admins = conn.execute("SELECT count(*) FROM users WHERE lower(role)='admin'").fetchone()[0]
        tot_locked = conn.execute("SELECT count(*) FROM users WHERE account_status='locked' OR failed_attempts>=3").fetchone()[0]

    s1, s2, s3 = st.columns(3)
    with s1:
        st.markdown(
            f'<div class="pn-card" style="text-align:center;">'
            f'<span style="font-size:24px;">👥</span><br>'
            f'<span style="font-size:12px;color:{COLORS["text_muted"]};text-transform:uppercase;">Total Registered Users</span>'
            f'<h2 style="margin:10px 0 0;color:{COLORS["accent_gold"]};">{tot_users}</h2>'
            f'</div>',
            unsafe_allow_html=True
        )
    with s2:
        st.markdown(
            f'<div class="pn-card" style="text-align:center;">'
            f'<span style="font-size:24px;">🔐</span><br>'
            f'<span style="font-size:12px;color:{COLORS["text_muted"]};text-transform:uppercase;">Administrators</span>'
            f'<h2 style="margin:10px 0 0;color:{COLORS["accent_gold"]};">{tot_admins}</h2>'
            f'</div>',
            unsafe_allow_html=True
        )
    with s3:
        st.markdown(
            f'<div class="pn-card" style="text-align:center;">'
            f'<span style="font-size:24px;">📈</span><br>'
            f'<span style="font-size:12px;color:{COLORS["text_muted"]};text-transform:uppercase;">Locked Accounts</span>'
            f'<h2 style="margin:10px 0 0;color:{COLORS["red"]};">{tot_locked}</h2>'
            f'</div>',
            unsafe_allow_html=True
        )

    st.markdown("---")
    search_q = st.text_input("🔍 Search User Registry", placeholder="Enter name or email...").strip()
    users = backend.get_all_users(search_q)
    
    if not users:
        st.info("No matching accounts found.")
    else:
        users_df = pd.DataFrame(users, columns=["ID", "Username", "Email", "Role", "Failed Tries", "Status"])
        for index, row in users_df.iterrows():
            c1, c2, c3, c4 = st.columns([2, 1, 1.2, 0.8])
            with c1:
                st.markdown(f"**{row['Username']}** ({row['Email']})<br>Status: `{row['Status']}` | Failed Attempts: `{row['Failed Tries']}`", unsafe_allow_html=True)
            with c2:
                roles_list = ["Logistics Manager", "Pricing Analyst", "Carrier Auditor", "Executive", "Admin"]
                current_role = row['Role']
                if current_role not in roles_list:
                    roles_list.append(current_role)
                selected_role = st.selectbox("Change Role", roles_list, index=roles_list.index(current_role), key=f"role_sel_{row['ID']}")
                if selected_role != current_role:
                    backend.update_role(row['ID'], selected_role)
                    st.success(f"Updated {row['Username']} to {selected_role}!")
                    st.rerun()
            with c3:
                if row['Status'] == 'locked' or row['Failed Tries'] >= 3:
                    if st.button("🔓 Unlock", key=f"unlock_{row['ID']}", use_container_width=True):
                        backend.unlock_user(row['ID'])
                        st.success(f"Unlocked {row['Username']}!")
                        st.rerun()
            with c4:
                if row['Email'] != "infosys@ai" and row['Username'] != "admin" and row['Username'] != st.session_state.get("username"):
                    st.markdown('<div class="red-btn-container">', unsafe_allow_html=True)
                    if st.button("🗑️ Delete", key=f"del_{row['ID']}", use_container_width=True):
                        backend.delete_user(row['ID'])
                        st.success(f"Deleted user {row['Username']}!")
                        st.rerun()
                    st.markdown('</div>', unsafe_allow_html=True)
            st.divider()

    st.subheader("➕ Create New User Account")
    with st.form("admin_add_user_form", clear_on_submit=True):
        a_username = st.text_input("Username").strip()
        a_email = st.text_input("Email").strip()
        a_pw = st.text_input("Initial Password", type="password")
        a_role = st.selectbox("Role", ["Logistics Manager", "Pricing Analyst", "Carrier Auditor", "Executive", "Admin"])
        submit_btn = st.form_submit_button("Create User Account")
        if submit_btn:
            if a_username and a_email and a_pw:
                try:
                    backend.add_user(a_username, a_email, a_pw, a_role)
                    st.success(f"✅ User {a_username} created successfully!")
                    st.rerun()
                except Exception:
                    st.error("❌ Failed: Username or Email may already exist.")
            else:
                st.warning("Please fill out all fields.")

def render_system_settings():
    st.markdown('<h2 style="margin-top:0;color:white;">⚙️ System Settings & Environment</h2>', unsafe_allow_html=True)
    render_card(
        '<h3>⚙️ System Configuration & Secrets Status</h3>'
        '<p style="color:#94a3b8;font-size:13px;margin:0;">Inspect runtime secrets, vector database indexing status, database parameters, and environment flags.</p>'
    )
    col1, col2 = st.columns(2)
    with col1:
        st.subheader("🔒 Environment Credentials")
        st.markdown(f"- **JWT Secret**: `{'Configured' if config_utils.JWT_SECRET_KEY else 'Missing'}`")
        st.markdown(f"- **HuggingFace Token**: `{'Loaded' if config_utils.HF_TOKEN else 'Not Provided'}`")
        st.markdown(f"- **SMTP Email Address**: `{config_utils.EMAIL_ADDRESS or 'Not Provided'}`")
        st.markdown(f"- **Admin Email**: `{config_utils.ADMIN_EMAIL_ID}`")
        st.markdown(f"- **Ngrok Auth Token**: `{'Configured' if config_utils.NGROK_AUTHTOKEN else 'Missing'}`")
    with col2:
        st.subheader("📦 Database & Vector Store Settings")
        st.markdown(f"- **SQLite DB Path**: `{config_utils.DATABASE}`")
        st.markdown(f"- **RAG Documents Folder**: `{config_utils.RAG_DIR}`")
        st.markdown(f"- **FAISS Index Folder**: `{config_utils.FAISS_DIR}`")
        st.markdown(f"- **Embedding Model**: `{config_utils.EMBEDDING_MODEL_NAME}`")
        st.markdown(f"- **LLM Engine**: `{config_utils.QWEN_MODEL}`")

def render_admin_dashboard():
    st.markdown('<h2 style="margin-top:0;color:white;">🛡️ System Administration Console</h2>', unsafe_allow_html=True)
    tab_users, tab_metrics, tab_settings = st.tabs(["User Lifecycle Management", "🚚 ML Model Card Registry", "⚙️ System Configuration"])
    
    with tab_users:
        render_user_management()
    with tab_metrics:
        render_card(
            '<h3>🚚 Professional ML Model Card Registry</h3>'
            '<p style="color:#94a3b8;font-size:13px;margin:0;">Dynamic transparency portal showing champion models, training metrics, and local statuses.</p>'
        )
        
        a1_m = backend.get_latest_model_metrics("Agent1_Pricing")
        a1_status = "Loaded" if os.path.exists(config_utils.AGENT1_MODEL_PATH) else "Standby"
        
        a2_m = backend.get_latest_model_metrics("Agent2_DelayRisk")
        a2_status = "Loaded" if os.path.exists(config_utils.AGENT2_MODEL_PATH) else "Standby"
        
        a3_m = backend.get_latest_model_metrics("Agent3_CarrierCompliance")
        a3_status = "Loaded" if os.path.exists(config_utils.AGENT3_MODEL_PATH) else "Standby"

        m_col1, m_col2, m_col3 = st.columns(3)
        with m_col1:
            st.markdown(
                f'<div class="pn-card" style="border-top:4px solid {COLORS["border_gold"]};">'
                f'<h4 style="margin:0 0 6px;">💰 Agent 1: Pricing</h4>'
                f'<div style="font-size:12px;margin-bottom:8px;">Status: <b>{a1_status}</b></div>'
                f'Champion: <b>{a1_m[0] if a1_m else "HistGradientBoostingRegressor"}</b><br>'
                f'R² Score: <b>{a1_m[1]:.4f}</b> (Threshold ≥ 0.90)<br>'
                f'RMSE: <b>${a1_m[2]:,.2f}</b><br>'
                f'Training Samples: <b>{a1_m[4] if a1_m else "3,000"}</b>'
                f'</div>',
                unsafe_allow_html=True
            )
        with m_col2:
            st.markdown(
                f'<div class="pn-card" style="border-top:4px solid {COLORS["accent_orange"]};">'
                f'<h4 style="margin:0 0 6px;">🚢 Agent 2: Delay</h4>'
                f'<div style="font-size:12px;margin-bottom:8px;">Status: <b>{a2_status}</b></div>'
                f'Champion: <b>{a2_m[0] if a2_m else "RandomForestClassifier"}</b><br>'
                f'Accuracy: <b>{a2_m[3]*100:.1f}%</b><br>'
                f'Training Samples: <b>{a2_m[4] if a2_m else "3,000"}</b>'
                f'</div>',
                unsafe_allow_html=True
            )
        with m_col3:
            st.markdown(
                f'<div class="pn-card" style="border-top:4px solid {COLORS["green"]};">'
                f'<h4 style="margin:0 0 6px;">✅ Agent 3: Compliance</h4>'
                f'<div style="font-size:12px;margin-bottom:8px;">Status: <b>{a3_status}</b></div>'
                f'Champion: <b>{a3_m[0] if a3_m else "ExtraTreesClassifier"}</b><br>'
                f'Accuracy: <b>{a3_m[3]*100:.1f}%</b><br>'
                f'Training Samples: <b>{a3_m[4] if a3_m else "3,000"}</b>'
                f'</div>',
                unsafe_allow_html=True
            )
    with tab_settings:
        render_system_settings()

# =============================================================================
# 4. APP INITIALIZATION & SECURITY GATE
# =============================================================================
apply_theme()
backend.start_background_warmup()

# Check authentication
if not st.session_state.get("token"):
    render_auth_portal()
    st.stop()

# Retrieve user sessions
username = st.session_state.get("username", "guest")
user_role = st.session_state.get("role", "Logistics Manager")
is_admin = user_role.lower() == "admin"

# Global Translation Preference
def translate_if_needed(text):
    target_lang = st.session_state.get("global_target_lang", "English")
    if target_lang != "English":
        tgt_code = backend.resolve_flores_code(target_lang)
        return backend.translate_text(text, src_lang="eng_Latn", tgt_lang=tgt_code)
    return text

# Get list of ports from single source of truth
ports_list = config_utils.get_port_list()

# =============================================================================
# 5. NAVIGATION LAYOUT & ROUTING
# =============================================================================
nav_options = [
    "🏠 Dashboard",
    "🤖 AI Copilot (RAG)",
    "💰 Agent 1: Pricing",
    "🚢 Agent 2: Route & Weather",
    "✅ Agent 3: Carrier Audit",
    "📜 Prediction History",
    "🔄 Retrain Pipeline"
]
nav_icons = [
    "grid-1x2-fill",
    "cpu-fill",
    "cash-coin",
    "geo-alt-fill",
    "shield-check",
    "clock-history",
    "arrow-repeat"
]

if is_admin:
    nav_options.extend([
        "🛡️ Admin Dashboard",
        "👥 User Management",
        "⚙️ Settings"
    ])
    nav_icons.extend([
        "sliders",
        "people-fill",
        "gear-fill"
    ])

nav_options.append("🚪 Logout")
nav_icons.append("box-arrow-right")

default_idx = 0
if st.session_state.get("admin_redirect") and is_admin:
    if "🛡️ Admin Dashboard" in nav_options:
        default_idx = nav_options.index("🛡️ Admin Dashboard")
    st.session_state["admin_redirect"] = False

with st.sidebar:
    st.markdown(
        f'<div style="text-align:center;padding:12px 0 6px;font-weight:800;font-size:20px;'
        f'color:{COLORS["accent"]};letter-spacing:0.5px;">⚡ FreightQuote AI</div>',
        unsafe_allow_html=True
    )
    st.markdown(
        f'<div style="text-align:center;font-size:12px;color:{COLORS["text_body"]};'
        f'margin-bottom:18px;">Active Session: <b>{username}</b><br>'
        f'<span style="background:{COLORS["bg_alt"]};padding:2px 8px;border-radius:4px;'
        f'color:{COLORS["accent"]};font-weight:600;font-size:11px;">[{user_role}]</span></div>',
        unsafe_allow_html=True
    )

    selected_tab = option_menu(
        menu_title=None,
        options=nav_options,
        icons=nav_icons,
        default_index=default_idx,
        styles={
            "container": {"padding": "0!important", "background-color": "transparent"},
            "nav-link": {
                "font-size": "13px",
                "text-align": "left",
                "margin": "3px 0",
                "border-radius": "8px",
                "color": COLORS["text_body"],
                "font-weight": "500",
                "background-color": "transparent"
            },
            "nav-link-selected": {
                "background-color": COLORS["bg_card"],
                "color": COLORS["accent"],
                "border": f"1px solid {COLORS['accent']}",
                "font-weight": "700"
            },
        }
    )

    st.markdown("---")
    st.markdown("<div style='font-size:11px;font-weight:700;margin-bottom:4px;color:white;'>🌐 Translation Preferences</div>", unsafe_allow_html=True)
    global_lang = st.selectbox(
        "Display Language",
        options=["English", "Hindi", "Tamil", "Telugu", "Kannada", "Spanish", "French", "Arabic"],
        index=0,
        key="global_target_lang"
    )
    
    st.markdown("---")
    # Display model loading status in sidebar
    status = backend.get_model_status()
    st.markdown("<div style='font-size:12px;font-weight:700;margin-bottom:8px;color:white;'>🤖 Model Agent Status</div>", unsafe_allow_html=True)
    for agent, loaded in status.items():
        lbl = "Ready" if loaded else "Offline"
        color = COLORS["green"] if loaded else COLORS["red"]
        st.markdown(
            f'<div style="font-size:11px;margin-bottom:4px;">● {agent}: '
            f'<span style="color:{color};font-weight:600;">{lbl}</span></div>',
            unsafe_allow_html=True
        )
    st.markdown(
        f'<div style="font-size:11px;margin-top:8px;color:{COLORS["green"]};font-weight:600;">'
        f'● RAG Vector Store: Indexed</div>',
        unsafe_allow_html=True
    )

if selected_tab == "🚪 Logout":
    st.session_state["token"] = None
    st.session_state["username"] = None
    st.session_state["role"] = None
    st.session_state["auth_page"] = "login"
    st.rerun()

render_header("FreightQuote AI Portal", f"Platform Hub / {selected_tab}", icon="🧭")

# DB statistics context loader
with config_utils.get_connection() as conn:
    n_quotes = conn.execute("SELECT count(*) FROM quotes").fetchone()[0]
    n_shipments = conn.execute("SELECT count(*) FROM shipments").fetchone()[0]
    n_carriers = conn.execute("SELECT count(*) FROM carriers").fetchone()[0]
    n_alerts = conn.execute("SELECT count(*) FROM notifications").fetchone()[0]
    n_users = conn.execute("SELECT count(*) FROM users").fetchone()[0]
    n_admins = conn.execute("SELECT count(*) FROM users WHERE lower(role)='admin'").fetchone()[0]

active_models_cnt = sum(1 for val in status.values() if val)
db_stats = {
    "total_quotes": n_quotes, "total_shipments": n_shipments, "total_carriers": n_carriers,
    "total_alerts": n_alerts, "total_users": n_users, "total_admins": n_admins,
    "active_models": active_models_cnt
}

# =============================================================================
# TAB MODULES
# =============================================================================
if selected_tab == "🏠 Dashboard":
    st.subheader(translate_if_needed("📊 Executive Operations Dashboard"))
    kpi_cols = st.columns(5)
    
    with kpi_cols[0]:
        st.markdown(
            f'<div class="pn-card" style="text-align:center;padding:15px;">'
            f'<span style="font-size:11px;color:{COLORS["text_muted"]};">{translate_if_needed("Ports Monitored")}</span>'
            f'<h2 style="margin:4px 0;color:{COLORS["accent"]};">{len(ports_list)}</h2>'
            f'<span style="font-size:9px;color:{COLORS["green"]};font-weight:700;">🟢 {translate_if_needed("Active Corridors")}</span>'
            f'</div>',
            unsafe_allow_html=True
        )
    with kpi_cols[1]:
        st.markdown(
            f'<div class="pn-card" style="text-align:center;padding:15px;">'
            f'<span style="font-size:11px;color:{COLORS["text_muted"]};">{translate_if_needed("Active Quotes")}</span>'
            f'<h2 style="margin:4px 0;color:{COLORS["accent_orange"]};">{n_quotes}</h2>'
            f'<span style="font-size:9px;color:{COLORS["text_muted"]};">{translate_if_needed("SQLite Logged")}</span>'
            f'</div>',
            unsafe_allow_html=True
        )
    with kpi_cols[2]:
        st.markdown(
            f'<div class="pn-card" style="text-align:center;padding:15px;">'
            f'<span style="font-size:11px;color:{COLORS["text_muted"]};">{translate_if_needed("Active ML Models")}</span>'
            f'<h2 style="margin:4px 0;color:{COLORS["green"]};">{active_models_cnt} / 3</h2>'
            f'<span style="font-size:9px;color:{COLORS["green"]};font-weight:700;">🟢 {translate_if_needed("Evaluated")}</span>'
            f'</div>',
            unsafe_allow_html=True
        )
    with kpi_cols[3]:
        llm_lbl = "Active" if backend.is_llm_loaded() else "Standby"
        llm_color = COLORS["green"] if backend.is_llm_loaded() else COLORS["yellow"]
        st.markdown(
            f'<div class="pn-card" style="text-align:center;padding:15px;">'
            f'<span style="font-size:11px;color:{COLORS["text_muted"]};">{translate_if_needed("LLM GPU Status")}</span>'
            f'<h2 style="margin:4px 0;color:{llm_color};">{translate_if_needed(llm_lbl)}</h2>'
            f'<span style="font-size:9px;color:{COLORS["text_muted"]};">Qwen2.5-3B</span>'
            f'</div>',
            unsafe_allow_html=True
        )
    with kpi_cols[4]:
        st.markdown(
            f'<div class="pn-card" style="text-align:center;padding:15px;">'
            f'<span style="font-size:11px;color:{COLORS["text_muted"]};">{translate_if_needed("RAG FAISS + BM25")}</span>'
            f'<h2 style="margin:4px 0;color:{COLORS["green"]};">{translate_if_needed("Hybrid")}</h2>'
            f'<span style="font-size:9px;color:{COLORS["green"]};font-weight:700;">🟢 {translate_if_needed("Ready")}</span>'
            f'</div>',
            unsafe_allow_html=True
        )

    st.markdown("---")
    col_l, col_r = st.columns([1.5, 1])
    with col_l:
        st.subheader(translate_if_needed("🌐 AI & Network Health Status"))
        net_col1, net_col2 = st.columns(2)
        with net_col1:
            st.markdown(
                f'<div class="pn-card-alt" style="border-left: 5px solid {COLORS["green"]};">'
                f'<h5 style="margin:0 0 4px;font-size:13px;">{translate_if_needed("Global Network Health")}</h5>'
                f'<h3 style="margin:0;color:{COLORS["green"]};">{translate_if_needed("NORMAL")}</h3>'
                f'<p style="margin:4px 0 0;font-size:11px;color:{COLORS["text_muted"]};">{translate_if_needed("All corridors operating within typical latency bounds.")}</p>'
                f'</div>',
                unsafe_allow_html=True
            )
        with net_col2:
            high_risk_ports = [p for p, data in config_utils.PORTS.items() if data["risk"] == "High"]
            st.markdown(
                f'<div class="pn-card-alt" style="border-left: 5px solid {COLORS["red"]};">'
                f'<h5 style="margin:0 0 4px;font-size:13px;">{translate_if_needed("High-Risk Alerts")}</h5>'
                f'<h3 style="margin:0;color:{COLORS["red"]};">{len(high_risk_ports)} {translate_if_needed("Ports")}</h3>'
                f'<p style="margin:4px 0 0;font-size:11px;color:{COLORS["text_muted"]};">{translate_if_needed("Active monitoring in: ")} {", ".join(high_risk_ports)}</p>'
                f'</div>',
                unsafe_allow_html=True
            )
            
        st.markdown(f"#### {translate_if_needed('AI Agent Health Matrix')}")
        faiss_status = "🟢 Ready" if os.path.exists(config_utils.FAISS_DIR) else "🔴 Offline"
        bm25_status = "🟢 Ready" if os.path.exists(os.path.join(config_utils.FAISS_DIR, "bm25_index.pkl")) else "🟡 Missing sparse index"
        sqlite_status = "🟢 Normal"
        try:
            with config_utils.get_connection() as c: c.execute("SELECT 1")
        except Exception: sqlite_status = "🔴 Offline"
        
        weather_status = "🟢 API Online"
        try:
            res = backend.get_live_weather(18.95, 72.82)
            if res["status"] != "success": weather_status = "🟡 Degraded"
        except Exception: weather_status = "🔴 Offline"
            
        health_data = [
            {"Component": "Qwen2.5-3B-Instruct (LLM)", "Status": "🟢 Active (GPU)" if backend.is_llm_loaded() else "🟡 Standby (CPU Fallback)"},
            {"Component": "NLLB-200 (Translation)", "Status": "🟢 Active" if backend.is_nllb_ready() else "🟡 Standby"},
            {"Component": "FAISS Index (Dense RAG)", "Status": faiss_status},
            {"Component": "BM25 Index (Sparse RAG)", "Status": bm25_status},
            {"Component": "SQLite Database", "Status": sqlite_status},
            {"Component": "Open-Meteo Weather API", "Status": weather_status}
        ]
        st.dataframe(pd.DataFrame(health_data), use_container_width=True, hide_index=True)

    with col_r:
        st.subheader(translate_if_needed("🔔 Active System Notifications"))
        with config_utils.get_connection() as conn:
            alerts = conn.execute("SELECT subject, message, created_at FROM notifications ORDER BY id DESC LIMIT 5").fetchall()
        if not alerts:
            st.info(translate_if_needed("No alerts logged in the notification registry."))
        else:
            for subject, msg, dt in alerts:
                st.markdown(
                    f'<div style="background:{COLORS["bg_card"]};padding:8px 12px;border-radius:6px;'
                    f'border:1px solid {COLORS["border"]};margin-bottom:8px;font-size:12px;">'
                    f'<div style="font-weight:bold;color:{COLORS["accent"]};">{translate_if_needed(subject)}</div>'
                    f'<div style="color:{COLORS["text_body"]};margin:4px 0;">{translate_if_needed(msg)}</div>'
                    f'<div style="color:{COLORS["text_muted"]};font-size:10px;">{dt}</div></div>',
                    unsafe_allow_html=True
                )

    st.markdown("---")
    st.subheader(translate_if_needed("🚢 Recent Database Activity"))
    col_dl, col_dr = st.columns(2)
    with col_dl:
        st.markdown(f"##### {translate_if_needed('Logged Quotes')}")
        with config_utils.get_connection() as conn:
            quotes_df = pd.read_sql_query("SELECT quote_id, origin, destination, final_cost_usd, created_at FROM quotes ORDER BY created_at DESC LIMIT 5", conn)
        if quotes_df.empty: st.info("No quotes recorded yet.")
        else: st.dataframe(quotes_df, use_container_width=True, hide_index=True)
    with col_dr:
        st.markdown(f"##### {translate_if_needed('Active Shipments')}")
        with config_utils.get_connection() as conn:
            shipments_df = pd.read_sql_query("SELECT shipment_id, carrier_name, actual_cost, status FROM shipments ORDER BY created_at DESC LIMIT 5", conn)
        if shipments_df.empty: st.info("No active shipments found.")
        else: st.dataframe(shipments_df, use_container_width=True, hide_index=True)

elif selected_tab == "🤖 AI Copilot (RAG)":
    render_card(
        f'<h3>🤖 {translate_if_needed("Grounded Logistics Copilot (Hybrid RAG)")}</h3>'
        f'<p style="color:#94a3b8;font-size:13px;margin:0;">{translate_if_needed("Interactive RAG Knowledge Base Engine. Combines dense semantic search (FAISS) and sparse lexical search (BM25) to provide verifiable, grounded responses.")}</p>'
    )
    cfg_col1, cfg_col2 = st.columns(2)
    with cfg_col1: st.session_state["faiss_weight"] = st.slider("FAISS Dense Weight", 0.0, 1.0, 0.6, step=0.1)
    with cfg_col2: st.session_state["bm25_weight"] = st.slider("BM25 Sparse Weight", 0.0, 1.0, 0.4, step=0.1)

    with st.form("rag_copilot_form", clear_on_submit=False):
        rag_query_input = st.text_input(
            translate_if_needed("Ask Grounded Logistics Copilot a Question:"),
            placeholder="e.g. What are the seller duties under FOB Incoterm 2020? Or how is volumetric weight calculated?"
        )
        ask_btn = st.form_submit_button(translate_if_needed("🔍 Ask Copilot"), use_container_width=True)

    if ask_btn and rag_query_input.strip():
        detected_lang_code = backend.detect_language(rag_query_input)
        detected_lang = [k for k, v in backend.NLLB_LANGS.items() if v == detected_lang_code][0]
        global_target = st.session_state.get("global_target_lang", "English")
        requires_translation = (global_target != "English") or (detected_lang_code != "eng_Latn")
        
        with st.spinner("Executing hybrid RAG pipeline & generating grounded answer..."):
            if detected_lang_code != "eng_Latn":
                english_query = backend.translate_text(rag_query_input, src_lang=detected_lang_code, tgt_lang="eng_Latn")
            else:
                english_query = rag_query_input
                
            rag_result = backend.execute_rag_query(english_query)
            if global_target != "English":
                display_lang_code = backend.resolve_flores_code(global_target)
                display_answer = backend.translate_text(rag_result["answer"], src_lang="eng_Latn", tgt_lang=display_lang_code)
            elif detected_lang_code != "eng_Latn":
                display_answer = backend.translate_text(rag_result["answer"], src_lang="eng_Latn", tgt_lang=detected_lang_code)
            else:
                display_answer = rag_result["answer"]
                
        badge_style = COLORS["green"] if "HIGH" in rag_result["grounding_level"] else (COLORS["yellow"] if "LIMITED" in rag_result["grounding_level"] else COLORS["red"])
        st.markdown(f"### 💡 {translate_if_needed('Grounded Answer')}")
        st.markdown(f'<div class="pn-card" style="border-left:5px solid {badge_style};">{display_answer}</div>', unsafe_allow_html=True)
                    
        st.markdown(f"#### 🛡️ {translate_if_needed('AI Evidence & Grounding Panel')}")
        ev_col1, ev_col2, ev_col3 = st.columns(3)
        with ev_col1:
            st.markdown(f"**⚡ {translate_if_needed('Grounding Level')}**: <span style='color:{badge_style};font-weight:700;'>{rag_result['grounding_level']}</span>", unsafe_allow_html=True)
            st.caption(translate_if_needed(rag_result["confidence_desc"]))
        with ev_col2:
            st.markdown(f"**⏱️ {translate_if_needed('Response Latency')}**: `{rag_result['latency_sec']} seconds`")
            st.markdown(f"**📌 {translate_if_needed('Knowledge Category')}**: `{rag_result['top_category']}`")
        with ev_col3:
            val_lbl = "PASSED" if rag_result["validation"]["is_valid"] else "WARNING"
            val_col = COLORS["green"] if rag_result["validation"]["is_valid"] else COLORS["red"]
            st.markdown(f"**🛡️ {translate_if_needed('Evidence Check')}**: <span style='color:{val_col};font-weight:700;'>{val_lbl}</span>", unsafe_allow_html=True)
            if not rag_result["validation"]["is_valid"]:
                st.caption(f"Hallucinated values filtered: {rag_result['validation']['unsupported_numbers']}")

        if requires_translation and "translation_quality" in st.session_state:
            tq = st.session_state["translation_quality"]
            tq_col = COLORS["green"] if tq["is_valid"] else COLORS["red"]
            st.markdown(f"##### 🌐 {translate_if_needed('Translation Quality Validation Report')}")
            st.markdown(
                f'<div style="background:{COLORS["bg_alt"]};border:1px solid {COLORS["border"]};padding:12px;border-radius:6px;font-size:12px;">'
                f'Language: <b>{tq["language"]}</b> | Engine: <b>{tq["engine"]}</b> | Technical terms preserved: <b>{tq["terms_preserved"]}</b> | '
                f'Numbers preserved: <b>{"✓" if tq["numbers_preserved"] else "✗"}</b> | Currency preserved: <b>{"✓" if tq["currency_preserved"] else "✗"}</b> | '
                f'Units preserved: <b>{"✓" if tq["units_preserved"] else "✗"}</b> | Status: <span style="color:{tq_col};font-weight:bold;">{tq["status"]}</span>'
                f'</div>', unsafe_allow_html=True
            )

        st.markdown("---")
        st.subheader(translate_if_needed("📖 Retrieved Vector Store Chunks & Source Citations"))
        for idx, cit in enumerate(rag_result["citations"], 1):
            st.markdown(
                f'<div class="pn-card-alt">'
                f'<span class="pn-badge" style="color:{COLORS["green"]};border-color:{COLORS["green"]};font-weight:bold;margin-bottom:8px;">🔵 RAG DOCUMENT</span><br>'
                f'<b>Source #{idx}: {cit["source"]}</b> (Page {cit["page"]} of {cit["total_pages"]} | Category: <code>{cit["category"]}</code>)<br>'
                f'Chunk ID: <code>{cit["chunk_id"]}</code> | Combined Hybrid Score: <span style="color:{COLORS["accent_gold"]};font-weight:bold;">{cit["score"]:.4f}</span><br>'
                f'<span style="font-size:11px;color:{COLORS["text_muted"]};">FAISS Dense: {cit["dense_score"]:.3f} | BM25 Sparse: {cit["sparse_score"]:.3f}</span>'
                f'<hr style="margin:8px 0;border-color:{COLORS["border"]};">'
                f'<p style="font-size:13px;color:{COLORS["text_body"]};margin:0;"><i>"{cit["snippet"]}"</i></p>'
                f'</div>', unsafe_allow_html=True
            )

    st.markdown("---")
    with st.expander(translate_if_needed("📂 Knowledge Base Manager (Google Drive / Local RAG Docs)"), expanded=False):
        st.subheader(translate_if_needed("Knowledge Base Configuration"))
        pdf_files = [f for f in os.listdir(config_utils.RAG_DIR) if f.endswith(".pdf")]
        vs = backend.get_vector_store()
        num_chunks = len(vs.docstore._dict) if vs else 0
        
        stat_col1, stat_col2, stat_col3 = st.columns(3)
        with stat_col1: st.metric(translate_if_needed("Total PDF Manuals"), len(pdf_files))
        with stat_col2: st.metric(translate_if_needed("Indexed Text Chunks"), num_chunks)
        with stat_col3: st.metric(translate_if_needed("FAISS Status"), "Indexed" if vs else "Standby")
            
        st.markdown("---")
        st.subheader(translate_if_needed("Upload New Knowledge PDF"))
        uploaded_file = st.file_uploader("Select a PDF document to upload:", type="pdf")
        if uploaded_file is not None:
            save_path = os.path.join(config_utils.RAG_DIR, uploaded_file.name)
            with open(save_path, "wb") as f:
                f.write(uploaded_file.getbuffer())
            st.success(f"Uploaded {uploaded_file.name} to {config_utils.RAG_DIR}!")
            log_alert("System", "Document Uploaded", f"Uploaded new PDF: {uploaded_file.name}")
            
        st.markdown("---")
        st.subheader(translate_if_needed("Rebuild Knowledge Index"))
        col_btn1, col_btn2 = st.columns(2)
        with col_btn1:
            if st.button("🔄 Refresh Knowledge Base", use_container_width=True):
                backend.clear_bm25_cache()
                st.success("Cleared BM25 cache successfully!")
                st.rerun()
        with col_btn2:
            if st.button("🚨 Rebuild RAG Indices (FAISS + BM25)", use_container_width=True):
                with st.spinner("Rebuilding FAISS and BM25 index from scratch..."):
                    backend.clear_bm25_cache()
                    if os.path.exists(os.path.join(config_utils.FAISS_DIR, "index.faiss")):
                        os.remove(os.path.join(config_utils.FAISS_DIR, "index.faiss"))
                    if os.path.exists(os.path.join(config_utils.FAISS_DIR, "index.pkl")):
                        os.remove(os.path.join(config_utils.FAISS_DIR, "index.pkl"))
                    backend.get_vector_store()
                st.success("Successfully rebuilt FAISS and BM25 RAG index!")
                log_alert("System", "Index Rebuilt", "FAISS + BM25 RAG indexes rebuilt.")
                st.rerun()
                
        st.markdown(f"##### {translate_if_needed('Available Documents in RAG Folder')}")
        st.dataframe(pd.DataFrame(pdf_files, columns=["Filename"]), use_container_width=True, hide_index=True)

elif selected_tab == "💰 Agent 1: Pricing":
    render_card(
        f'<h3>💰 {translate_if_needed("Agent 1: Global Freight Price Advisor")}</h3>'
        f'<p style="color:#94a3b8;font-size:13px;margin:0;">{translate_if_needed("Calculates base freight quotes by checking distance, weight, port dwell, and congestion indices.")}</p>'
    )
    c1, c2 = st.columns(2)
    with c1:
        st.subheader(translate_if_needed("Inputs"))
        origin_port = st.selectbox("Origin Port", ports_list, index=0)
        dest_port = st.selectbox("Destination Port", ports_list, index=4)
        orig_details = config_utils.get_port_details(origin_port)
        dest_details = config_utils.get_port_details(dest_port)
        
        if origin_port == dest_port:
            calculated_dist = 100.0
        else:
            lat1, lon1 = np.radians(orig_details["latitude"]), np.radians(orig_details["longitude"])
            lat2, lon2 = np.radians(dest_details["latitude"]), np.radians(dest_details["longitude"])
            calculated_dist = round(2 * np.arcsin(np.sqrt(np.sin((lat2 - lat1)/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1)/2)**2)) * 3440.065, 1)
            
        dist = st.number_input("Route Distance (nm)", min_value=100.0, max_value=25000.0, value=float(calculated_dist))
        weight = st.number_input("Cargo Weight (tons)", min_value=0.5, max_value=1000.0, value=45.0)
        cong_level_idx = ["Low", "Medium", "High"].index(dest_details["congestion"])
        cong_level = st.selectbox("Congestion Surcharge Level", ["Low (0)", "Medium (1)", "High (2)"], index=cong_level_idx)
        fuel_idx = st.slider("Fuel Surcharge Index", 0.8, 1.8, 1.15)
        cargo_type = st.selectbox("Cargo Category", ["General (0)", "Perishable (1)", "Hazmat (2)", "Heavy Lift (3)"], index=0)
        dwell_days = st.number_input("Port Dwell Cooldown (days)", min_value=0.1, max_value=20.0, value=float(orig_details["dwell_time"]))

        cong_v = int(cong_level.split("(")[1].replace(")", ""))
        cargo_v = int(cargo_type.split("(")[1].replace(")", ""))
        calculate = st.button(translate_if_needed("💰 Calculate Cost Quote"))

    with c2:
        st.subheader(translate_if_needed("Quotation Output"))
        if calculate:
            mean_cost, lo_cost, hi_cost = backend.predict_freight(dist, weight, cong_v, fuel_idx, cargo_v, dwell_days)
            breakdown = backend.get_quote_breakdown(mean_cost, dist, weight, cong_v, fuel_idx, cargo_v, dwell_days)

            st.markdown(
                f'<div class="pn-card" style="border-top:4px solid {COLORS["accent"]};text-align:center;">'
                f'<h4 style="margin:0 0 8px;color:{COLORS["text_heading"]};">🤖 Agent 1: {translate_if_needed("Pricing Estimator")}</h4>'
                f'<div style="font-size:11px;color:{COLORS["text_muted"]};margin-bottom:12px;">Status: <span class="pn-badge status-ready">{translate_if_needed("Ready")}</span></div>'
                f'<h1 style="color:{COLORS["accent_gold"]};margin:12px 0 6px;">${mean_cost:,.2f}</h1>'
                f'<p style="margin:0;font-size:14px;color:{COLORS["text_body"]};">'
                f'95% {translate_if_needed("Confidence Interval")}: <b>${lo_cost:,.2f} — ${hi_cost:,.2f}</b></p>'
                f'</div>', unsafe_allow_html=True
            )
            
            st.markdown(f"#### 🔍 {translate_if_needed('Why is this quote expensive?')}")
            breakdown_data = [
                {"Cost Component": translate_if_needed("Base Freight"), "Charge (USD)": f"${breakdown['Base Freight']:,.2f}"},
                {"Cost Component": translate_if_needed("BAF (Fuel Surcharge)"), "Charge (USD)": f"${breakdown['BAF']:,.2f}"},
                {"Cost Component": translate_if_needed("Customs Fees"), "Charge (USD)": f"${breakdown['Customs']:,.2f}"},
                {"Cost Component": translate_if_needed("Terminal Handling"), "Charge (USD)": f"${breakdown['Terminal']:,.2f}"},
                {"Cost Component": translate_if_needed("Marine Insurance"), "Charge (USD)": f"${breakdown['Insurance']:,.2f}"},
                {"Cost Component": "<b>" + translate_if_needed("Final Quote") + "</b>", "Charge (USD)": f"<b>${breakdown['Total']:,.2f}</b>"}
            ]
            st.write(pd.DataFrame(breakdown_data).to_html(escape=False, index=False), unsafe_allow_html=True)
            st.markdown(f"<div style='margin-top:10px;font-size:13px;'><b>{translate_if_needed('Primary Cost Driver')}:</b> <span style='color:{COLORS["accent_orange"]};font-weight:700;'>{translate_if_needed(breakdown['Main Driver'])}</span></div>", unsafe_allow_html=True)
            
            with st.spinner("Generating pricing explanation..."):
                sys_p = "You are the FreightQuote AI Pricing explainability agent. Explain why this quote has the calculated cost, explaining the impact of the primary cost driver using plain professional business language."
                user_p = (
                    f"Breakdown: Base={breakdown['Base Freight']}, BAF={breakdown['BAF']}, Customs={breakdown['Customs']}, Terminal={breakdown['Terminal']}, Insurance={breakdown['Insurance']}\n"
                    f"Primary driver: {breakdown['Main Driver']}\n"
                    f"Write a 2-sentence explanation of the primary cost driver."
                )
                explanation = backend.generate_grounded_answer(sys_p, user_p, max_tokens=150)
                st.markdown(f'<div class="pn-card-alt" style="margin-top:10px;"><i>"{translate_if_needed(explanation)}"</i></div>', unsafe_allow_html=True)
                
            st.markdown(f"#### 📌 {translate_if_needed('Data Provenance & Sources')}")
            prov_data = [
                {"Parameter/Metric": translate_if_needed("Port coordinates & details"), "Data Source": "🟡 DATABASE (config_utils.PORTS)"},
                {"Parameter/Metric": translate_if_needed("Route distance calculations"), "Data Source": "💜 ML SOLVER (Haversine Formula)"},
                {"Parameter/Metric": translate_if_needed("Base cost predictions"), "Data Source": "💜 ML PREDICTION (Agent 1 Regressor Model)"},
                {"Parameter/Metric": translate_if_needed("Cost breakdown math"), "Data Source": "🟡 DATABASE (Deterministic Business Logic)"},
                {"Parameter/Metric": translate_if_needed("Explainability narrative"), "Data Source": "🔵 RAG DOCUMENT (Qwen Grounded Synthesis)"}
            ]
            st.dataframe(pd.DataFrame(prov_data), use_container_width=True, hide_index=True)

            with config_utils.get_connection() as conn:
                q_id = f"Q-{int(datetime.datetime.now().timestamp())}"
                conn.execute("""
                INSERT INTO quotes (quote_id, created_by, origin, destination, distance_nm, weight_tons,
                                    shipment_mode, port_congestion, cargo_type, base_cost_usd, margin_usd,
                                    adjustment_factor, final_cost_usd, delay_risk_prob, risk_summary, audit_flag)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """, (q_id, username, origin_port, dest_port, dist, weight, "Sea", str(cong_v), str(cargo_v),
                      mean_cost * 0.8, mean_cost * 0.2, fuel_idx, mean_cost, 0.45, f"Calculated route: {origin_port} to {dest_port}", "Passed"))
                conn.commit()
            log_alert("In-App", "Freight Quote Generated", f"New freight quotation generated: ID {q_id} for ${mean_cost:,.2f}")
            st.success(f"{translate_if_needed('Quotation')} {q_id} {translate_if_needed('logged successfully in SQLite database.')}")

elif selected_tab == "🚢 Agent 2: Route & Weather":
    render_card(
        f'<h3>🚢 {translate_if_needed("Agent 2: Route Delay & Marine Weather Sentinel")}</h3>'
        f'<p style="color:#94a3b8;font-size:13px;margin:0;">{translate_if_needed("Predicts transit latency using live weather, port berth constraints, and canal transit backlogs.")}</p>'
    )
    c1, c2 = st.columns(2)
    with c1:
        st.subheader(translate_if_needed("Route Configuration"))
        origin = st.selectbox("Origin Port", ports_list, index=0)
        destination = st.selectbox("Destination Port", ports_list, index=4)
        orig_details = config_utils.get_port_details(origin)
        dest_details = config_utils.get_port_details(destination)
        
        st.markdown(f"##### 🌦️ {translate_if_needed('Live Weather Information')}")
        orig_weather = backend.get_live_weather(orig_details["latitude"], orig_details["longitude"])
        dest_weather = backend.get_live_weather(dest_details["latitude"], dest_details["longitude"])
        
        w_cols = st.columns(2)
        with w_cols[0]:
            st.markdown(
                f'<div style="background:{COLORS["bg_alt"]};border:1px solid {COLORS["border"]};padding:12px;border-radius:6px;">'
                f'<b>{origin}</b> ({orig_weather["source"]})<br>'
                f'Temp: <b>{orig_weather["temperature_c"]}°C</b><br>'
                f'Wind: <b>{orig_weather["wind_speed_kmh"]} km/h</b><br>'
                f'Condition: <b>{orig_weather["condition"]}</b><br>'
                f'Risk: <b>{orig_weather["risk_level"]}</b>'
                f'</div>', unsafe_allow_html=True
            )
        with w_cols[1]:
            st.markdown(
                f'<div style="background:{COLORS["bg_alt"]};border:1px solid {COLORS["border"]};padding:12px;border-radius:6px;">'
                f'<b>{destination}</b> ({dest_weather["source"]})<br>'
                f'Temp: <b>{dest_weather["temperature_c"]}°C</b><br>'
                f'Wind: <b>{dest_weather["wind_speed_kmh"]} km/h</b><br>'
                f'Condition: <b>{dest_weather["condition"]}</b><br>'
                f'Risk: <b>{dest_weather["risk_level"]}</b>'
                f'</div>', unsafe_allow_html=True
            )
            
        dwell = st.slider("Historical Dwell (days)", 0.5, 15.0, float(orig_details["dwell_time"]))
        berth_cap = st.slider("Port Berth Count", 5, 50, int(dest_details["vessel_count"] // 2 if dest_details["vessel_count"] > 10 else 10))
        canal_queue = st.checkbox("Active Canal Bottlenecks?", value=True if "suez" in origin.lower() or "suez" in destination.lower() or "said" in origin.lower() or "said" in destination.lower() else False)
        season = st.selectbox("Seasonal Risk Profile", ["Normal Route Conditions (0.2)", "Monsoon Squalls (0.5)", "North Sea Winter Latency (0.7)"])
        season_v = float(season.split("(")[1].replace(")", ""))
        test_delay = st.button(translate_if_needed("🚢 Predict Route Delay Risk"))

    with c2:
        st.subheader(translate_if_needed("Risk Analytics"))
        if test_delay:
            route_len = 8600.0
            weather_v = (orig_weather["risk_score"] + dest_weather["risk_score"]) / 2.0
            prob, lo, hi = backend.predict_delay(dwell, berth_cap, route_len, weather_v, int(canal_queue), season_v)

            badge_color = COLORS["red"] if prob > 0.60 else (COLORS["yellow"] if prob > 0.35 else COLORS["green"])
            risk_lbl = "HIGH RISK" if prob > 0.60 else ("MODERATE RISK" if prob > 0.35 else "LOW RISK")
            st.markdown(
                f'<div class="pn-card" style="border-top:4px solid {badge_color};text-align:center;">'
                f'<h4 style="margin:0 0 8px;color:{COLORS["text_heading"]};">🤖 Agent 2: {translate_if_needed("Delay Sentinel")}</h4>'
                f'<div style="font-size:11px;color:{COLORS["text_muted"]};margin-bottom:12px;">Status: <span class="pn-badge status-ready">{translate_if_needed("Ready")}</span></div>'
                f'<h2 style="color:white;margin:12px 0 6px;">{prob * 100:.1f}% {translate_if_needed("Delay Probability")} ({translate_if_needed(risk_lbl)})</h2>'
                f'<p style="margin:0;font-size:14px;">95% {translate_if_needed("Confidence Interval")}: <b>{lo*100:.1f}% — {hi*100:.1f}%</b></p>'
                f'</div>', unsafe_allow_html=True
            )

            categories = ["Port Dwell Impact", "Berth Constraints", "Canal Dependency", "Seasonal Risk Factors", "Overall Score"]
            values = [dwell / 15 * 10, (50 - berth_cap) / 50 * 10, 10.0 if canal_queue else 2.0, season_v * 10, prob * 10]
            fig = go.Figure()
            fig.add_trace(go.Scatterpolar(
                r=values + [values[0]], theta=categories + [categories[0]], fill='toself',
                line_color=COLORS["accent"], fillcolor='rgba(245, 158, 11, 0.2)'
            ))
            fig.update_layout(
                polar=dict(radialaxis=dict(visible=True, range=[0, 10])), showlegend=False,
                paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
                height=250, margin=dict(l=40, r=40, t=20, b=20)
            )
            st.plotly_chart(fig, use_container_width=True)
            
            st.markdown(f"#### 📌 {translate_if_needed('Data Provenance & Sources')}")
            prov_data = [
                {"Parameter/Metric": translate_if_needed("Port coordinates & names"), "Data Source": "🟡 DATABASE (config_utils.PORTS)"},
                {"Parameter/Metric": translate_if_needed("Live weather temperatures / wind"), "Data Source": "🟢 LIVE API DATA (Open-Meteo API)"},
                {"Parameter/Metric": translate_if_needed("Marine transit risk predictions"), "Data Source": "💜 ML PREDICTION (Agent 2 Delay Classifier)"}
            ]
            st.dataframe(pd.DataFrame(prov_data), use_container_width=True, hide_index=True)

    st.markdown("---")
    st.subheader(translate_if_needed("🚢 Route Comparison Utility"))
    st.markdown(translate_if_needed("Compare multiple logistics routes side-by-side to optimize transit speed, cost, and safety."))
    
    comp_cols = st.columns(3)
    with comp_cols[0]:
        r1_orig = st.selectbox("Route A Origin", ports_list, index=0)
        r1_dest = st.selectbox("Route A Destination", ports_list, index=4)
    with comp_cols[1]:
        r2_orig = st.selectbox("Route B Origin", ports_list, index=1)
        r2_dest = st.selectbox("Route B Destination", ports_list, index=7)
    with comp_cols[2]:
        r3_orig = st.selectbox("Route C Origin", ports_list, index=15)
        r3_dest = st.selectbox("Route C Destination", ports_list, index=7)
        
    if st.button("🚢 Run Side-by-Side Comparison", use_container_width=True):
        compare_rows = []
        for route_name, o, d in [("Route A", r1_orig, r1_dest), ("Route B", r2_orig, r2_dest), ("Route C", r3_orig, r3_dest)]:
            o_details = config_utils.get_port_details(o)
            d_details = config_utils.get_port_details(d)
            
            lat1, lon1 = np.radians(o_details["latitude"]), np.radians(o_details["longitude"])
            lat2, lon2 = np.radians(d_details["latitude"]), np.radians(d_details["longitude"])
            route_dist = round(2 * np.arcsin(np.sqrt(np.sin((lat2 - lat1)/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1)/2)**2)) * 3440.065, 1)
            
            weather_risk_orig = backend.get_live_weather(o_details["latitude"], o_details["longitude"])["risk_level"]
            weather_risk_dest = backend.get_live_weather(d_details["latitude"], d_details["longitude"])["risk_level"]
            weather_risk = "High" if "High" in [weather_risk_orig, weather_risk_dest] else ("Medium" if "Medium" in [weather_risk_orig, weather_risk_dest] else "Low")
            congestion_level = "High" if "High" in [o_details["congestion"], d_details["congestion"]] else ("Medium" if "Medium" in [o_details["congestion"], d_details["congestion"]] else "Low")
            
            prob, _, _ = backend.predict_delay(
                float(o_details["dwell_time"]), 20, float(route_dist), 
                0.4 if weather_risk == "Medium" else (0.8 if weather_risk == "High" else 0.2), 
                1 if "suez" in o.lower() or "suez" in d.lower() else 0, 
                0.5 if weather_risk == "High" else 0.2
            )
            mean_cost, _, _ = backend.predict_freight(route_dist, 45.0, 1 if congestion_level == "Medium" else (2 if congestion_level == "High" else 0), 1.15, 0, float(o_details["dwell_time"]))
            rec = "⚠️ Watch" if prob > 0.50 or weather_risk == "High" else "✅ Recommended"
            
            compare_rows.append({
                "Route": route_name, "Origin Port": o, "Destination Port": d,
                "Distance (nm)": f"{route_dist:,.1f}", "Port Congestion": congestion_level,
                "Live Weather Risk": weather_risk, "Delay Probability": f"{prob*100:.1f}%",
                "Est. Freight Cost": f"${mean_cost:,.2f}", "Status/Recommendation": rec
            })
        st.write(pd.DataFrame(compare_rows).to_html(escape=False, index=False), unsafe_allow_html=True)

elif selected_tab == "✅ Agent 3: Carrier Audit":
    render_card(
        f'<h3>✅ {translate_if_needed("Agent 3: Carrier Compliance Sentinel & Tariff Auditor")}</h3>'
        f'<p style="color:#94a3b8;font-size:13px;margin:0;">{translate_if_needed("Audits registered carriers for tariff compliance, delays, and documentation logs.")}</p>'
    )
    with config_utils.get_connection() as conn:
        carriers_df = pd.read_sql_query("SELECT * FROM carriers", conn)

    if carriers_df.empty:
        st.warning("No carrier entries found. Initialize databases to seed samples.")
    else:
        col_l, col_r = st.columns([1.2, 1])
        with col_l:
            st.subheader(translate_if_needed("Carrier Registry & Scorecards"))
            for index, row in carriers_df.iterrows():
                punct = row["punctuality_rate"] * 100
                compliance = row["tariff_compliance_score"] * 100
                delay_penalty = max(0, 100 - (row["avg_delay_days"] * 10))
                surcharge_penalty = max(0, 100 - (row["fuel_surcharge_pct"] * 3))
                overall = round((punct + compliance + delay_penalty + surcharge_penalty) / 4)
                
                badge_style = COLORS["green"] if overall >= 85 else (COLORS["yellow"] if overall >= 70 else COLORS["red"])
                flagged_status = "🚨 FLAGGED" if row["flagged"] else "Active"
                flag_style = COLORS["red"] if row["flagged"] else COLORS["green"]
                
                st.markdown(
                    f'<div style="background:{COLORS["bg_card"]};border:1px solid {COLORS["border"]};padding:15px;border-radius:6px;margin-bottom:12px;">'
                    f'<div style="display:flex;justify-content:between;align-items:center;">'
                    f'<h4 style="margin:0;font-size:14px;color:white;">{row["carrier_name"]} (ID: {row["carrier_id"]})</h4>'
                    f'<span style="background:{badge_style};padding:2px 8px;border-radius:4px;color:black;font-weight:bold;font-size:11px;">{overall}% Score</span>'
                    f'</div>'
                    f'<div style="font-size:11px;color:{COLORS["text_muted"]};margin-top:4px;">'
                    f'Mode: <b>{row["transport_mode"]}</b> | Delay: <b>{row["avg_delay_days"]} days</b> | Fuel Surcharge: <b>{row["fuel_surcharge_pct"]}%</b> | Tier: <b>{row["tier_rating"]}</b> | Status: <b style="color:{flag_style};">{flagged_status}</b>'
                    f'</div>'
                    f'</div>', unsafe_allow_html=True
                )

        with col_r:
            st.subheader(translate_if_needed("Compliance Audit Panel"))
            c_select = st.selectbox("Select Carrier for Audit", carriers_df["carrier_name"].tolist())
            c_row = carriers_df[carriers_df["carrier_name"] == c_select].iloc[0]

            punct_v = float(c_row["punctuality_rate"])
            delay_v = float(c_row["avg_delay_days"])
            fuel_s_v = float(c_row["fuel_surcharge_pct"])
            tariff_v = float(c_row["tariff_compliance_score"])
            prob, lo, hi = backend.predict_compliance(punct_v, delay_v, 0.05, fuel_s_v, tariff_v, 1)

            badge_color = COLORS["green"] if prob > 0.85 else (COLORS["yellow"] if prob > 0.65 else COLORS["red"])
            flag_status = "🚨 FLAGGED" if int(c_row.get("flagged", 0)) else "Passed"
            st.markdown(
                f'<div class="pn-card" style="border-top:4px solid {badge_color};text-align:center;">'
                f'<h4 style="margin:0 0 8px;color:{COLORS["text_heading"]};">🤖 Agent 3: {translate_if_needed("Compliance Auditor")}</h4>'
                f'<div style="font-size:11px;color:{COLORS["text_muted"]};margin-bottom:12px;">Status: <span class="pn-badge status-ready">{translate_if_needed("Ready")}</span></div>'
                f'<h2 style="color:white;margin:12px 0 6px;">{prob * 100:.1f}% {translate_if_needed("Compliance Score")}</h2>'
                f'<p style="margin:0;font-size:13px;">95% CI: <b>{lo*100:.1f}% — {hi*100:.1f}%</b> | {translate_if_needed("Audit Status")}: <b>{flag_status}</b></p>'
                f'</div>', unsafe_allow_html=True
            )

            col_b1, col_b2 = st.columns(2)
            with col_b1:
                is_currently_flagged = int(c_row.get("flagged", 0))
                btn_lbl = translate_if_needed("Clear Flag") if is_currently_flagged else translate_if_needed("🚨 Flag Carrier")
                if st.button(btn_lbl, key="flag_carrier_btn", use_container_width=True):
                    with config_utils.get_connection() as conn:
                        conn.execute("UPDATE carriers SET flagged=? WHERE carrier_id=?", (0 if is_currently_flagged else 1, c_row["carrier_id"]))
                        conn.commit()
                    log_alert("In-App", "Carrier Flag Updated", f"Carrier {c_select} flag status updated.")
                    st.rerun()
            with col_b2:
                if st.button(translate_if_needed("📋 Generate Audit Report"), key="gen_json_btn", use_container_width=True):
                    with st.spinner("Synthesizing audit metrics..."):
                        report = backend.orchestrate_3_agents_query(
                            f"Create a compliance audit advisory note for carrier {c_select}",
                            {"base_rate_usd": 15000}, {"delay_risk_pct": int(delay_v * 10)},
                            {"compliance": "Flagged" if is_currently_flagged else "Passed"}, db_stats=db_stats
                        )
                    st.markdown(f"### {translate_if_needed('Generative Audit Report')}")
                    st.markdown(translate_if_needed(report))
                    
            st.markdown(f"#### 📌 {translate_if_needed('Data Provenance & Sources')}")
            prov_data = [
                {"Parameter/Metric": translate_if_needed("Carrier baseline statistics"), "Data Source": "🟡 DATABASE (carriers table)"},
                {"Parameter/Metric": translate_if_needed("Carrier compliance predictions"), "Data Source": "💜 ML PREDICTION (Agent 3 Compliance Model)"},
                {"Parameter/Metric": translate_if_needed("Audit text generation"), "Data Source": "🔵 RAG DOCUMENT (Qwen Grounded Synthesis)"}
            ]
            st.dataframe(pd.DataFrame(prov_data), use_container_width=True, hide_index=True)

elif selected_tab == "📜 Prediction History":
    st.subheader("📜 System Prediction History Registry")
    tab_hist1, tab_hist2 = st.tabs(["📑 Logged Price Quotations", "🚢 Active/Delivered Shipments"])

    with tab_hist1:
        with config_utils.get_connection() as conn:
            quotes_full_df = pd.read_sql_query("SELECT * FROM quotes ORDER BY created_at DESC", conn)
        if quotes_full_df.empty: st.info("No price quotations logged yet.")
        else:
            st.dataframe(quotes_full_df, use_container_width=True, hide_index=True)
            csv_data = quotes_full_df.to_csv(index=False).encode('utf-8')
            st.download_button(label="📥 Download Quotation CSV Log", data=csv_data, file_name="freightquote_quotes_history.csv", mime="text/csv", use_container_width=True)
    with tab_hist2:
        with config_utils.get_connection() as conn:
            shipments_full_df = pd.read_sql_query("SELECT * FROM shipments ORDER BY created_at DESC", conn)
        if shipments_full_df.empty: st.info("No active shipments registered in database.")
        else:
            st.dataframe(shipments_full_df, use_container_width=True, hide_index=True)
            csv_ship_data = shipments_full_df.to_csv(index=False).encode('utf-8')
            st.download_button(label="📥 Download Shipments CSV Log", data=csv_ship_data, file_name="freightquote_shipments_history.csv", mime="text/csv", use_container_width=True)

elif selected_tab == "🔄 Retrain Pipeline":
    st.subheader("🔄 Multi-Agent Retraining Dashboard")
    col_l, col_r = st.columns([1, 1.5])
    with col_l:
        render_card(
            '<h4>🔄 Pipeline Trigger</h4>'
            '<p style="color:#94a3b8;font-size:12px;">Trigger a multi-algorithm retrain loop. '
            'Compares 6 models and updates SQLite metrics card.</p>'
        )
        if st.button("🔄 Execute Pipeline Retrain", use_container_width=True):
            with st.spinner("Retraining ML agents (matching 6 algorithms policy)..."):
                import subprocess
                res = subprocess.run(["python", "train_ml.py"], capture_output=True, text=True)
                if res.returncode == 0: st.success("✅ Retraining complete! Metrics loaded into SQLite database.")
                else:
                    st.error("❌ Retraining process failed.")
                    st.code(res.stderr)
            st.rerun()
    with col_r:
        st.subheader("📜 ML model History")
        with config_utils.get_connection() as conn:
            try:
                hist_df = pd.read_sql_query("SELECT agent_name, model_name, r2_score, accuracy, created_at FROM ml_models ORDER BY id DESC", conn)
                if hist_df.empty: st.info("No training records in database yet.")
                else: st.dataframe(hist_df, use_container_width=True, hide_index=True)
            except Exception: st.info("No model history table initialized yet.")

    st.markdown("---")
    st.subheader("🔔 In-App Notifications & Alerts Log")
    with config_utils.get_connection() as conn:
        alerts = conn.execute("SELECT recipient, subject, message, created_at FROM notifications ORDER BY id DESC LIMIT 10").fetchall()
    if not alerts: st.info("No alerts logged in the notification registry.")
    else:
        for recipient, subject, msg, dt in alerts:
            st.markdown(
                f'<div style="background:{COLORS["bg_card"]};padding:8px 16px;border-radius:6px;'
                f'border:1px solid {COLORS["border"]};margin-bottom:8px;font-size:13px;">'
                f'<span style="color:{COLORS["accent"]}; font-weight:bold;">[{recipient.upper()}]</span> '
                f'<b>{subject}</b> — {msg} <span style="float:right;color:{COLORS["text_muted"]};font-size:11px;">{dt}</span></div>',
                unsafe_allow_html=True
            )

elif selected_tab == "🛡️ Admin Dashboard":
    if not is_admin: st.error("🔒 Security Gate: Admin privileges required to load this console.")
    else: render_admin_dashboard()
elif selected_tab == "👥 User Management":
    if not is_admin: st.error("🔒 Security Gate: Admin privileges required to load this console.")
    else: render_user_management()
elif selected_tab == "⚙️ Settings":
    if not is_admin: st.error("🔒 Security Gate: Admin privileges required to load this console.")
    else: render_system_settings()

# Universal Unified Footer
st.markdown("---")
st.markdown(
    f'<div style="text-align:center;font-size:12px;color:{COLORS["text_muted"]};padding:20px 0 10px;">'
    f'<b>FreightQuote AI Enterprise Portal</b><br>'
    f'Enterprise Multi-Agent Logistics Platform | Infosys Springboard Milestone 4 Consolidated<br>'
    f'Developed by Sai Laghuvar'
    f'</div>',
    unsafe_allow_html=True
)

### Step 6: Write Retrain Pipeline Script
Writes the standalone training script `train_ml.py` to the working folder.

In [ ]:
%%writefile /content/freightquote_m4/train_ml.py
"""
train_ml.py — FreightQuote AI Multi-Agent Training Pipeline.
Compares 6 distinct machine learning algorithms per agent.
Saves best model checkpoints and logs metrics to the SQLite ml_models table.
Completely local and self-contained (no Kaggle credentials or network lookups required).
"""
import os
import joblib
import sqlite3
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, roc_auc_score, accuracy_score
from sklearn.calibration import CalibratedClassifierCV

# Regressors
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, AdaBoostRegressor

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.svm import SVC

from config_utils import (DATABASE, AGENT1_MODEL_PATH, AGENT2_MODEL_PATH, AGENT3_MODEL_PATH,
                           get_connection, save_ml_metrics, init_db)

# -----------------------------
# Multi-Algorithm Evaluators
# -----------------------------
def compare_regressors(models_dict, X_tr, X_te, y_tr, y_te, agent_name, save_path):
    print(f"\n[EVALUATING] Regressors for {agent_name}:")
    best_name, best_model, best_r2 = None, None, -np.inf

    for name, model in models_dict.items():
        try:
            model.fit(X_tr, y_tr)
            preds = model.predict(X_te)
            r2 = float(r2_score(y_te, preds))
            rmse = float(np.sqrt(mean_squared_error(y_te, preds)))
            print(f"  - {name:30s} R2 = {r2:.4f} | RMSE = {rmse:,.2f}")

            # Save all evaluator metrics to history
            save_ml_metrics(agent_name, name, r2, rmse, 0.0, len(y_tr) + len(y_te), save_path)

            if r2 > best_r2:
                best_r2, best_name, best_model = r2, name, model
        except Exception as e:
            print(f"  - {name:30s} Failed: {e}")

    print(f"[CHAMPION] {best_name} (R2 = {best_r2:.4f})")
    joblib.dump(best_model, save_path)
    return best_model, best_name, best_r2

def compare_classifiers(models_dict, X_tr, X_te, y_tr, y_te, agent_name, save_path):
    print(f"\n[EVALUATING] Classifiers for {agent_name}:")
    best_name, best_model, best_auc = None, None, -np.inf

    for name, base_model in models_dict.items():
        try:
            model = CalibratedClassifierCV(base_model, cv=2, method="sigmoid")
            model.fit(X_tr, y_tr)

            probs = model.predict_proba(X_te)[:, 1]
            preds = model.predict(X_te)
            auc = float(roc_auc_score(y_te, probs))
            acc = float(accuracy_score(y_te, preds))
            print(f"  - {name:30s} ROC-AUC = {auc:.4f} | Accuracy = {acc*100:.1f}%")

            # Save metrics
            save_ml_metrics(agent_name, name, auc, 0.0, acc, len(y_tr) + len(y_te), save_path)

            if auc > best_auc:
                best_auc, best_name, best_model = auc, name, model
        except Exception as e:
            print(f"  - {name:30s} Failed: {e}")

    print(f"[CHAMPION] {best_name} (ROC-AUC = {best_auc:.4f})")
    joblib.dump(best_model, save_path)
    return best_model, best_name, best_auc

# -----------------------------
# Data Generator
# -----------------------------
def prepare_training_data(n_samples=2000, seed=42):
    init_db()
    rng = np.random.default_rng(seed)

    # 1. Agent 1 Pricing data generation
    weights = rng.uniform(10.0, 500.0, n_samples)
    a1_data = pd.DataFrame({
        "distance": rng.uniform(500.0, 15000.0, n_samples),
        "weight": weights,
        "congestion": rng.choice([0, 1, 2], n_samples, p=[0.4, 0.4, 0.2]),
        "fuel": rng.uniform(0.9, 1.6, n_samples),
        "cargo_type": rng.choice([0, 1, 2, 3], n_samples),
        "port_dwell": rng.uniform(0.5, 10.0, n_samples)
    })
    a1_data["target"] = (a1_data["distance"] * 1.8 + a1_data["weight"] * 48.0 + a1_data["congestion"] * 1600.0) * a1_data["fuel"] + rng.normal(0.0, 300.0, n_samples)

    # 2. Agent 2 Delay data generation
    dwell_vals = rng.uniform(0.5, 12.0, n_samples)
    a2_data = pd.DataFrame({
        "dwell": dwell_vals,
        "berth": rng.integers(5, 50, n_samples),
        "route_length": rng.uniform(500.0, 15000.0, n_samples),
        "weather": rng.uniform(0.0, 1.0, n_samples),
        "canal": rng.choice([0, 1], n_samples, p=[0.75, 0.25]),
        "season_risk": rng.uniform(0.1, 0.8, n_samples)
    })
    risk_score = a2_data["dwell"] / 12.0 * 0.4 + a2_data["weather"] * 0.3 + a2_data["canal"] * 0.2 + a2_data["season_risk"] * 0.1
    a2_data["delay_class"] = (risk_score > 0.50).astype(int)

    # 3. Agent 3 Compliance data generation
    punct_vals = rng.uniform(0.70, 0.99, n_samples)
    a3_data = pd.DataFrame({
        "punct": punct_vals,
        "avg_delay": rng.uniform(0.1, 5.0, n_samples),
        "complaint_rate": rng.uniform(0.0, 0.15, n_samples),
        "fuel_sc": rng.uniform(10.0, 20.0, n_samples),
        "tariff": rng.uniform(0.75, 1.0, n_samples),
        "docs_complete": rng.choice([0, 1], n_samples, p=[0.10, 0.90])
    })
    comp_score = a3_data["punct"] * 0.4 + a3_data["tariff"] * 0.3 + a3_data["docs_complete"] * 0.3 - a3_data["complaint_rate"] * 0.5
    a3_data["compliant"] = (comp_score > 0.65).astype(int)

    # Log merged records to SQLite database
    print("  [SQLITE] Storing 600 records in merged_datasets...")
    with get_connection() as conn:
        conn.execute("DELETE FROM merged_datasets")
        for i in range(min(600, n_samples)):
            conn.execute("""
            INSERT INTO merged_datasets (agent_target, dataset_source, origin, destination, distance_nm,
                                         weight_tons, freight_cost_usd, shipment_mode, port_congestion,
                                         dwell_time_days, berth_capacity, weather_disruption_level,
                                         carrier_punctuality, fuel_surcharge_pct, compliance_status)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                "All Agents", "SCMS+DataCo+Logistics+AuditData", "Mumbai", "Rotterdam",
                float(a1_data["distance"].iloc[i]), float(a1_data["weight"].iloc[i]),
                float(a1_data["target"].iloc[i]), "Ocean",
                ["Low", "Medium", "High"][int(a1_data["congestion"].iloc[i])],
                float(a2_data["dwell"].iloc[i]), int(a2_data["berth"].iloc[i]),
                float(a2_data["weather"].iloc[i]), float(a3_data["punct"].iloc[i]),
                float(a3_data["fuel_sc"].iloc[i]),
                "Compliant" if a3_data["compliant"].iloc[i] else "Flagged"
            ))
        conn.commit()
    print("  [OK] Data seeding complete.")
    return a1_data, a2_data, a3_data

# -----------------------------
# Train All Pipeline
# -----------------------------
def train_all_agents():
    print("=" * 60)
    print("[INFO] Running Multi-Agent Model Training Pipeline...")
    print("=" * 60)

    a1, a2, a3 = prepare_training_data()

    # ── AGENT 1: PRICING REGRESSOR (6 Algorithms) ──
    X1 = a1[["distance", "weight", "congestion", "fuel", "cargo_type", "port_dwell"]]
    y1 = a1["target"]
    X1_tr, X1_te, y1_tr, y1_te = train_test_split(X1, y1, test_size=0.2, random_state=42)

    regressors = {
        "Random Forest Regressor": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
        "Gradient Boosting Regressor": GradientBoostingRegressor(n_estimators=100, learning_rate=0.08, max_depth=4, random_state=42),
        "Extra Trees Regressor": ExtraTreesRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
        "Ridge Regressor": Pipeline([("scl", StandardScaler()), ("mdl", Ridge(alpha=1.0))]),
        "Decision Tree Regressor": DecisionTreeRegressor(max_depth=8, random_state=42),
        "AdaBoost Regressor": AdaBoostRegressor(n_estimators=50, random_state=42)
    }
    m1, name1, r2_1 = compare_regressors(regressors, X1_tr, X1_te, y1_tr, y1_te, "Agent1_Pricing", AGENT1_MODEL_PATH)
    print(f"Agent 1 R2 check: {'[PASS] (R2 >= 0.90)' if r2_1 >= 0.90 else '[WARNING] (R2 < 0.90)'}")

    # ── AGENT 2: ROUTE DELAY RISK CLASSIFIER (6 Algorithms) ──
    X2 = a2[["dwell", "berth", "route_length", "weather", "canal", "season_risk"]]
    y2 = a2["delay_class"]
    X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)

    classifiers_2 = {
        "Random Forest Classifier": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
        "Gradient Boosting Classifier": GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, max_depth=3, random_state=42),
        "Logistic Regression Classifier": Pipeline([("scl", StandardScaler()), ("mdl", LogisticRegression(max_iter=500, random_state=42))]),
        "Support Vector Classifier (RBF)": SVC(probability=True, random_state=42),
        "Extra Trees Classifier": ExtraTreesClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
        "AdaBoost Classifier": AdaBoostClassifier(n_estimators=50, random_state=42)
    }
    m2, name2, auc2 = compare_classifiers(classifiers_2, X2_tr, X2_te, y2_tr, y2_te, "Agent2_DelayRisk", AGENT2_MODEL_PATH)

    # ── AGENT 3: CARRIER COMPLIANCE SENTINEL (6 Algorithms) ──
    X3 = a3[["punct", "avg_delay", "complaint_rate", "fuel_sc", "tariff", "docs_complete"]]
    y3 = a3["compliant"]
    X3_tr, X3_te, y3_tr, y3_te = train_test_split(X3, y3, test_size=0.2, random_state=42, stratify=y3)

    classifiers_3 = {
        "Gradient Boosting Classifier": GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, max_depth=3, random_state=42),
        "Random Forest Classifier": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
        "Extra Trees Classifier": ExtraTreesClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
        "Logistic Regression Classifier": Pipeline([("scl", StandardScaler()), ("mdl", LogisticRegression(max_iter=500, random_state=42))]),
        "Decision Tree Classifier": DecisionTreeClassifier(max_depth=6, random_state=42),
        "AdaBoost Classifier": AdaBoostClassifier(n_estimators=50, random_state=42)
    }
    m3, name3, auc3 = compare_classifiers(classifiers_3, X3_tr, X3_te, y3_tr, y3_te, "Agent3_CarrierCompliance", AGENT3_MODEL_PATH)

    print("\n" + "=" * 60)
    print("[SUCCESS] Pipeline Training Complete!")
    print("=" * 60)
    print(f"Agent 1 Best: {name1:30s} | R2  = {r2_1:.4f}")
    print(f"Agent 2 Best: {name2:30s} | AUC = {auc2:.4f}")
    print(f"Agent 3 Best: {name3:30s} | AUC = {auc3:.4f}")
    print("=" * 60)

if __name__ == "__main__":
    train_all_agents()

### Step 7: Write Verification Script
Writes `test_m4_features.py` to check that the database, ports, weather APIs, translation engines, and hybrid RAG retrieval align perfectly.

In [ ]:
%%writefile /content/freightquote_m4/test_m4_features.py
"""
test_m4_features.py — Automated verification script for FreightQuote AI Milestone 4 enhancements.
"""
import sys
import os

# Set stdout to use UTF-8
sys.stdout.reconfigure(encoding='utf-8')

print("🚀 Starting Milestone 4 Verification Test Suite...")

# Test 1: Port Registry
print("\n--- Test 1: Port Registry (Single Source of Truth) ---")
try:
    import config_utils as ports
    port_list = ports.get_port_list()
    print(f"[PASS] Loaded ports registry. Total ports: {len(port_list)}")
    
    # Check details of Mundra & Suez
    mundra = ports.get_port_details("Mundra")
    suez = ports.get_port_details("Suez")
    
    if mundra and mundra["id"] == "INMUN" and mundra["country"] == "India":
        print(f"  - Mundra details: latitude={mundra['latitude']}, longitude={mundra['longitude']} [OK]")
    else:
        raise ValueError("Invalid details for Mundra")
        
    if suez and suez["id"] == "EGSUZ" and suez["risk"] == "High":
        print(f"  - Suez details: latitude={suez['latitude']}, congestion={suez['congestion']} [OK]")
    else:
        raise ValueError("Invalid details for Suez")
        
    print("[PASS] Port Registry verified.")
except Exception as e:
    print(f"[FAIL] Port Registry: {e}")

# Test 2: Live Weather Service
print("\n--- Test 2: Live Weather Service (Open-Meteo) ---")
try:
    import backend as weather_service
    # Test Mumbai JNPT coordinates
    mumbai_weather = weather_service.get_live_weather(18.95, 72.82)
    print(f"  - Weather Status: {mumbai_weather['status']}")
    print(f"  - Source Label: {mumbai_weather['source']}")
    print(f"  - Condition: {mumbai_weather['condition']}")
    print(f"  - Temperature: {mumbai_weather['temperature_c']}°C, Wind: {mumbai_weather['wind_speed_kmh']} km/h")
    print(f"  - Calculated Risk Level: {mumbai_weather['risk_level']}")
    
    if mumbai_weather["status"] in ["success", "fallback"]:
        print("[PASS] Weather Service verified.")
    else:
        raise ValueError("Failed weather response format")
except Exception as e:
    print(f"[FAIL] Weather Service: {e}")

# Test 3: Translation Quality & Terminology Guard
print("\n--- Test 3: Translation Quality & Terminology Guard ---")
try:
    import backend as translation_engine
    
    # Test terminology preservation check
    sample_eng = "The FOB price is calculated and includes a BAF surcharge of $500."
    # Simulated Hindi translation
    sample_hin = "FOB मूल्य की गणना की जाती है और इसमें $500 का BAF अधिभार शामिल है।"
    
    val = translation_engine.validate_translation(sample_eng, sample_hin, "hin_Deva")
    print(f"  - Validation status: {val['status']}")
    print(f"  - Numbers preserved: {val['numbers_preserved']} (Expected True)")
    print(f"  - Currency preserved: {val['currency_preserved']} (Expected True)")
    print(f"  - Terms preserved: {val['terms_preserved']} (Expected 2/2)")
    
    if val["is_valid"]:
        print("[PASS] Translation Validation verified.")
    else:
        raise ValueError("Translation validation failed on a correct translation")
except Exception as e:
    print(f"[FAIL] Translation Guard: {e}")

# Test 4: Hybrid RAG Retriever
print("\n--- Test 4: Hybrid FAISS + BM25 Retriever ---")
try:
    import backend as retriever
    # We should clear the cache and do a search
    retriever.clear_bm25_cache()
    
    query = "What is the difference between FOB and CIF Incoterms?"
    res = retriever.retrieve_context_and_citations(query, k=2)
    
    print(f"  - Citations retrieved: {len(res['citations'])}")
    for idx, cit in enumerate(res["citations"], 1):
        print(f"    Citation #{idx}: {cit['source']} (Page {cit['page']}) - Score: {cit['score']:.4f}")
        
    if len(res["citations"]) > 0:
        print("[PASS] Hybrid Retriever verified.")
    else:
        raise ValueError("No citations returned")
except Exception as e:
    print(f"[FAIL] Hybrid Retriever: {e}")

# Test 5: Grounding Pipeline and Refusal
print("\n--- Test 5: Grounding Pipeline & Refusal ---")
try:
    import backend as rag_engine
    
    # Test normal query
    query = "What are the seller responsibilities under FOB?"
    res = rag_engine.execute_rag_query(query)
    print(f"  - Grounding Level: {res['grounding_level']}")
    print(f"  - Response snippet: {res['answer'][:100]}...")
    
    # Test completely unsupported query (should refuse to answer)
    unsupported_query = "Who was the first president of the United States?"
    res_refusal = rag_engine.execute_rag_query(unsupported_query)
    print(f"  - Unsupported Query Grounding Level: {res_refusal['grounding_level']}")
    print(f"  - Refusal response: '{res_refusal['answer']}'")
    
    if "refuse" in res_refusal["answer"].lower() or "don't have enough" in res_refusal["answer"].lower() or "not available" in res_refusal["answer"].lower() or "insufficient" in res_refusal["grounding_level"].lower():
        print("[PASS] Grounding Pipeline Refusal verified.")
    else:
        raise ValueError("Grounding pipeline did not refuse unsupported query")
except Exception as e:
    print(f"[FAIL] Grounding Pipeline: {e}")

# Test 6: Cost Breakdown and Explainability
print("\n--- Test 6: Cost Breakdown and Explainability ---")
try:
    import backend as predict
    mean_cost, lo, hi = predict.predict_freight(8600.0, 45.0, 2, 1.15, 0, 3.5)
    breakdown = predict.get_quote_breakdown(mean_cost, 8600.0, 45.0, 2, 1.15, 0, 3.5)
    
    print(f"  - Base cost prediction: ${mean_cost:,.2f}")
    print(f"  - Base Freight portion: ${breakdown['Base Freight']:,.2f}")
    print(f"  - BAF: ${breakdown['BAF']:,.2f}")
    print(f"  - Customs: ${breakdown['Customs']:,.2f}")
    print(f"  - Terminal: ${breakdown['Terminal']:,.2f}")
    print(f"  - Insurance: ${breakdown['Insurance']:,.2f}")
    print(f"  - Main Driver: {breakdown['Main Driver']}")
    
    # Check sum matches exactly
    total_calc = sum([breakdown["Base Freight"], breakdown["BAF"], breakdown["Customs"], breakdown["Terminal"], breakdown["Insurance"]])
    diff = abs(total_calc - breakdown["Total"])
    print(f"  - Verification difference: ${diff:.4f}")
    
    if diff < 0.01:
        print("[PASS] Cost Breakdown verified.")
    else:
        raise ValueError("Sum of cost breakdown components does not match total cost")
except Exception as e:
    print(f"[FAIL] Cost Breakdown: {e}")

print("\n==================================================")
print("Milestone 4 Verification Complete!")
print("==================================================")

### Step 8: Run Automated Verification Tests
Compiles the newly written files and runs the verification test suite to confirm operational soundness.

In [ ]:
!python -m py_compile /content/freightquote_m4/config_utils.py /content/freightquote_m4/backend.py /content/freightquote_m4/app.py
import sys
sys.path.append('/content/freightquote_m4')
os.chdir('/content/freightquote_m4')
!python /content/freightquote_m4/test_m4_features.py

### Step 9: Start Streamlit & Tunnel Service
Launches Streamlit in the background, prints the external IP address (password for localtunnel), and exposes the app.

In [ ]:
print("==========================================================")
print("🔑 LOCALTUNNEL ACCESS PASSWORD / IP:")
!curl ipv4.icanhazip.com
print("==========================================================")
print("Starting Streamlit and Localtunnel...")
print("Click on the generated link (e.g. *.locall.net) and input the IP/Password printed above to log in.")
print("==========================================================")

# Start Streamlit in background and localtunnel on port 8501
!streamlit run /content/freightquote_m4/app.py --server.port 8501 & npx localtunnel --port 8501